#  Section 0: Introduction & Pipeline Overview

Welcome to the complete fine-tuning and reinforcement learning pipeline for the **Nemotron-30B** model.  
This notebook walks through the full training lifecycle — from supervised reasoning alignment to reinforcement optimization and final competition submission.

---

##  Objective

The mission of this pipeline is to transform the base **Nemotron-30B** model into a highly structured reasoning agent capable of producing:

- Accurate multi-step reasoning
- Stable chain-of-thought generation
- Competition-grade outputs
- Efficient inference using lightweight adapters

---

#  Training Pipeline Architecture

The workflow is divided into **three strategic phases**:

| Phase | Name | Purpose |
|---|---|---|
| **1** | Supervised Fine-Tuning (SFT) | Teach structured reasoning patterns using curated demonstrations |
| **2** | BAPO / GRPO Reinforcement Optimization | Reinforce correct reasoning behavior through policy optimization |
| **3** | Making Final Tweaks | for better inference and accuracy |

---

#  Notebook Flow

```text
                                                                                             EDA
                                                                                              ↓
                                                                                        SFT Fine-Tuning
                                                                                              ↓
                                                                                        GRPO/BAPO Reinforcement
                                                                                              ↓
                                                                                        Final tweaks for the adapter
                                                                                              ↓
                                                                                        Adapter Export
                                                                                              ↓
                                                                                        Competition Submission

In [ ]:
import numpy as np 
import pandas as pd 
import polars as pl
import matplotlib.pyplot as plot
import seaborn as sns
import os
import shutil, stat, glob
import kagglehub
import gc
import polars as pl
import re
import subprocess

sns.set_theme(style="whitegrid")
plot.rcParams['figure.figsize'] = (12, 6)

ptxas_fix_dir = "/kaggle/working/_ptxas_fix"
os.makedirs(ptxas_fix_dir, exist_ok=True)

ptxas_bins = glob.glob("/kaggle/usr/lib/**/ptxas*", recursive=True)
for src in ptxas_bins:
    if os.path.isfile(src):
        dst = os.path.join(ptxas_fix_dir, os.path.basename(src))
        shutil.copy2(src, dst)
        os.chmod(dst, 0o755)

# Monkeypatch subprocess.Popen to reroute all executions of read-only ptxas
_orig_Popen = subprocess.Popen
class PatchedPopen(_orig_Popen):
    def __init__(self, args, *pargs, **kwargs):
        if isinstance(args, (list, tuple)) and len(args) > 0 and isinstance(args[0], str):
            if "ptxas" in args[0] and "/kaggle/usr/lib/" in args[0]:
                args = list(args)
                args[0] = os.path.join(ptxas_fix_dir, os.path.basename(args[0]))
        elif isinstance(args, str):
            if "ptxas" in args and "/kaggle/usr/lib/" in args:
                args = re.sub(r'/kaggle/usr/lib/\S+/ptxas\S*', lambda m: os.path.join(ptxas_fix_dir, os.path.basename(m.group(0))), args)
        super().__init__(args, *pargs, **kwargs)

subprocess.Popen = PatchedPopen

if ptxas_bins: # I was getting some issues that is why 
    print(f" Copied & fixed {len(ptxas_bins)} ptxas binaries. Subprocess hook active.")

In [ ]:
import sys
import os
import glob
import subprocess
import site

print("Initiating Targeted Offline Installation...")
TARGET_ARMORY = "/kaggle/input/notebooks/banwait13/datasets-for-nemotron-banwait/**/*.whl"

for whl in sorted(glob.glob(TARGET_ARMORY, recursive=True)):
    try:
        subprocess.check_output(
            [sys.executable, "-m", "pip", "install", "--no-deps", "--upgrade", "--force-reinstall", "--quiet", whl],
            stderr=subprocess.STDOUT
        )
        # print(f"    Installed: {os.path.basename(whl)}")
    except subprocess.CalledProcessError as e:
        # Gracefully skips incompatible wheels (like wrong Python versions) without crashing
        error_output = e.output.decode('utf-8').strip().split('\n')[-1]
        # print(f"    SKIPPED:   {os.path.basename(whl)} — Reason: {error_output}")

import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# NVIDIA's custom Nemotron architecture code imports cutlass internally
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import importlib
import site
import sys

print("\n Invalidating Python's internal cache to force-load TRL...")

importlib.reload(site)
importlib.invalidate_caches()

local_path = f"/root/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"
if local_path not in sys.path:
    sys.path.insert(0, local_path)

print(" Cache cleared.")
OUTPUT_DIR = "/kaggle/working"
print("\n Environment armed and ready.")

In [ ]:
sns.set_theme(style="whitegrid")
plot.rcParams['figure.figsize'] = (12, 6)
SFT_PATH  = os.path.join("/kaggle/input/datasets/banwait13/lora-adapter/DATASETS/SFT_FLAWLESS_FINAL.csv")
GRPO_PATH = os.path.join("/kaggle/input/datasets/banwait13/lora-adapter/DATASETS/GRPO_FLAWLESS_FINAL.csv")
RAW_PATH  = os.path.join("/kaggle/input/datasets/banwait13/lora-adapter/DATASETS/DATA.csv")
PLOTS_DIR = "/kaggle/working/EDA_Plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
print("Loading Nemotron Datasets with Polars...")
try:
    df_sft = pl.read_csv(SFT_PATH)
    df_grpo = pl.read_csv(GRPO_PATH)
    df_types = pl.read_csv(RAW_PATH)

    print(" DATASET SUMMARY")
    print(f"Total SFT Samples:  {df_sft.height:,}")
    print(f"Total GRPO Samples: {df_grpo.height:,}")
except Exception as e:
    print(f" Error loading files: {e}")
    print("Check if the paths in [CELL 1] match your Kaggle dataset names.")

In [ ]:
print("\n Analyzing Reasoning Categories...")

try:
    if 'df_types' in locals():
        type_counts = df_types["puzzle_type"].value_counts().sort("count", descending=True)
        pdf_counts = type_counts.to_pandas()
        
        sns.set_theme(style="darkgrid", rc={
            "axes.facecolor": "#121212", 
            "figure.facecolor": "#121212",
            "axes.edgecolor": "#2a2a2a",
            "grid.color": "#2a2a2a",     
            "text.color": "white",
            "axes.labelcolor": "#888888", 
            "xtick.color": "#888888",
            "ytick.color": "#888888"
        })
        
        fig, ax = plot.subplots(figsize=(10, 5))
        
        neon_palette = sns.color_palette("blend:#00e5ff,#004b5c", n_colors=len(pdf_counts))
        
        sns.barplot(
            data=pdf_counts, 
            x="puzzle_type", 
            y="count", 
            palette=neon_palette,
            ax=ax,
            edgecolor="none" 
        )

        for container in ax.containers:
            ax.bar_label(container, padding=5, color='white', fontweight='bold', fontsize=10)
        
        plot.title("DISTRIBUTION OF REASONING CATEGORIES", color='white', fontsize=12, pad=15, fontweight='bold', letterspacing=1.5)
        plot.ylabel("SAMPLE COUNT", fontsize=9, fontweight='bold', labelpad=10)
        plot.xlabel("PUZZLE CATEGORY", fontsize=9, fontweight='bold', labelpad=10)
        
        plot.xticks(rotation=25, ha='right', fontweight='bold')
        sns.despine(left=True, bottom=True) 
        plot.tight_layout()
        plot.show()

except Exception as e:
    print(f" Could not generate puzzle type distribution: {e}")

In [ ]:
print("\nAnalyzing Token/Word Counts for VRAM Planning...")

try:
    df_sft = df_sft.with_columns([
        pl.col("instruction").str.split(" ").list.len().alias("instr_len"),
        pl.col("response").str.split(" ").list.len().alias("resp_len"),
    ])
    
    df_plot = df_sft.select(["instr_len", "resp_len"]).to_pandas()
    
    plot.figure(figsize=(12, 6))
    
    plot.subplot(1, 2, 1)
    sns.histplot(df_plot['instr_len'], bins=20, color="skyblue", kde=True)
    plot.title("Instruction Length (Words)")
    
    plot.subplot(1, 2, 2)
    sns.histplot(df_plot['resp_len'], bins=20, color="salmon", kde=True)
    plot.title("Response Length (CoT + Answer)")
    
    plot.tight_layout()
    plot.show()
except Exception as e:
    print(f" Could not generate sequence length analysis: {e}")

In [ ]:
print(" CORE INSIGHTS")
try:
    print(f"- Avg Instruction Length: {df_sft['instr_len'].mean():.1f} words")
    print(f"- Avg Response Length:    {df_sft['resp_len'].mean():.1f} words")
    if 'type_counts' in locals():
        print(f"- Most Common Category:   {type_counts['puzzle_type'][0]} ({type_counts['count'][0]} samples)")
except Exception as e:
    pass
print("\nSTRATEGY: Using high-density verified CoT format.")
print("Format: <think> reasoning </think>\\n\\nFinal Answer: \\boxed{answer}")

# Section 1: Fine-Tuning

This section covers the full fine-tuning process: loading the base model, attaching LoRA adapters, formatting and tokenizing the dataset, configuring the trainer, and running Phase 1 SFT training.

## Step 1: Load the Base Model and Tokenizer

The Nemotron-30B model is loaded in native BFloat16 precision with the following setup:

- **Device map set to `auto`** so PyTorch automatically distributes layers across available GPU VRAM.
- **Low CPU memory usage enabled** to stream weights directly to the GPU, avoiding the 16 GB system RAM bottleneck on Kaggle.
- After loading, all base model parameters are frozen. Only the LoRA adapter weights will be trained.
- A LoRA adapter (rank 16) is then attached, targeting the Mamba state projection layers (`in_proj` and `out_proj`). The MoE expert MLP layers are intentionally excluded because targeting them would create billions of trainable parameters and exhaust optimizer memory.

In [ ]:
# # ─── VRAM Budget: ~40 GB ───────────────────────────────────────────────────
# # Model (30B MoE BF16) ~28 GB loaded across device_map=auto
# # LoRA trainable params (r=16, 9 modules) ~0.8 GB
# # Activations + grad checkpoint (seq=2048) ~6 GB
# # Optimizer states (paged_adamw_8bit) ~2 GB
# # Safety headroom: ~3 GB free
# # ────────────────────────────────────────────────────────────────────────────

# gc.collect()
# torch.cuda.empty_cache()

# import os
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# torch.cuda.set_per_process_memory_fraction(0.92)

# MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# print("Loading Nemotron-30B in native BFloat16 (no quantization)...")
# print(f"   Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     device_map="auto",
#     trust_remote_code=True,
#     low_cpu_mem_usage=True,
#     torch_dtype=torch.bfloat16
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
# if tokenizer.unk_token is not None:
#     tokenizer.pad_token = tokenizer.unk_token
# else:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# # ── CHANGE 1: Reduced from 4096 → 2048 ──────────────────────────────────────
# # Halves activation VRAM. Prevents OOM during SFT backward pass.
# # The eval engine uses 8192 at inference — we recover full context there.
# MAX_SEQ_LEN = 2048

# print(" Base Model + Tokenizer loaded successfully (BFloat16).")
# vram_after_load = torch.cuda.max_memory_reserved() / 1024**3
# print(f"   VRAM after model load: {vram_after_load:.1f} GB")

# # Enable gradient checkpointing BEFORE LoRA wrap
# model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# # Freeze all base params
# for param in model.parameters():
#     param.requires_grad = False

# print("Initializing LoRA adapter...")

# # ── CHANGE 2: Expanded target_modules ───────────────────────────────────────
# # Original: only in_proj, out_proj, up_proj, down_proj (Mamba + MoE only)
# # Upgraded: full attention stack (q/k/v/o) + Mamba + MoE experts + lm_head
# # This matches the reference adapter architecture the eval engine expects.
# # r=16 kept — r=32 would OOM on 40 GB with these 9 modules.
# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=[
#         "q_proj", "k_proj", "v_proj", "o_proj",   # Full attention coverage
#         "in_proj", "out_proj",                      # Mamba SSM layers
#         "up_proj", "down_proj",                     # MoE expert projections
#         "lm_head",                                  # Output head
#     ],
#     lora_dropout=0.05,   # Slight dropout for generalization (was 0.0)
#     bias="none",
#     task_type=TaskType.CAUSAL_LM,
# )

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total     = sum(p.numel() for p in model.parameters())
# print(f"\n LoRA adapters attached + gradient checkpointing ON.")
# print(f"   Trainable: {trainable:,} ({100*trainable/total:.2f}% of total)")
# print(f"   Frozen:    {total - trainable:,}")

# gc.collect()
# torch.cuda.empty_cache()


## Step 2: Define the Reasoning Trace Format

Two prompt templates are defined here:

- **Reasoning template:** Used when the answer contains a chain-of-thought (CoT). Wraps the reasoning trace inside `<think>...</think>` tags and places the final answer in `\\boxed{}` format.
- **Non-reasoning template:** Used for direct answers with no CoT trace.

The `format_sample` function inspects each training example, detects which format applies, and assembles the final prompt string automatically.

In [ ]:
# import re

# # Strict Regex to capture \boxed{} and its contents (supports one level of nesting)
# BOXED_PATTERN = re.compile(r"\\boxed\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}")

# # --- System prompts ---
# REASONING_ON_SYSTEM  = "detailed thinking on"
# REASONING_OFF_SYSTEM = "detailed thinking off"

# REASONING_TEMPLATE = (
#     "<|im_start|>system\n"
#     "{system_prompt}<|im_end|>\n"
#     "<|im_start|>user\n"
#     "{question}<|im_end|>\n"
#     "<|im_start|>assistant\n"
#     "<think>\n"
#     "{chain_of_thought}\n"
#     "</think>\n"
#     "{final_answer}<|im_end|>"
# )
# NON_REASONING_TEMPLATE = (
#     "<|im_start|>system\n"
#     "{system_prompt}<|im_end|>\n"
#     "<|im_start|>user\n"
#     "{question}<|im_end|>\n"
#     "<|im_start|>assistant\n"
#     "{final_answer}<|im_end|>"
# )

# def normalize_columns(example):
#     """Translate any dataset's column names to standard 'question' and 'answer'."""
#     question = (example.get("problem") or example.get("question") or
#                 example.get("instruction") or example.get("prompt") or "")
#     answer = (example.get("solution") or example.get("response") or
#               example.get("answer") or example.get("output") or "")
#     return {"question": str(question).strip(), "answer": str(answer).strip()}

# def format_sample(example):
#     """Universal formatter — auto-detects the best template."""
#     q = example.get("question", "")
#     a = example.get("answer", "")

#     if not q or not a or len(a.strip()) < 5:
#         return {"text": "", "is_valid": False}

#     box_match = BOXED_PATTERN.search(a)
    
#     # THE FIX 1: Look for <thought> instead of <think> to match your CSV!
#     has_think = "<thought>" in a and "</thought>" in a

#     if box_match:
#         final_answer = box_match.group(0)
#         if has_think:
#             # THE FIX 2: Split by <thought> instead of <think>
#             cot = a.split("<thought>")[-1].split("</thought>")[0].strip()
#         else:
#             cot = a.split(final_answer)[0].strip()
            
#         # THE FIX 3: Remove the extra tags and the "Final Answer:" text so they don't duplicate
#         cot = cot.replace("<thought>", "").replace("</thought>", "").replace("Final Answer:", "").strip()
        
#         if cot:
#             text = REASONING_TEMPLATE.format(
#                 system_prompt=REASONING_ON_SYSTEM, question=q,
#                 chain_of_thought=cot, final_answer=final_answer)
#         else:
#             text = NON_REASONING_TEMPLATE.format(
#                 system_prompt=REASONING_OFF_SYSTEM, question=q,
#                 final_answer=final_answer)
#     elif has_think:
#         cot = a.split("<thought>")[-1].split("</thought>")[0].strip()
#         final_answer = a.split("</thought>")[-1].strip() or a
#         cot = cot.replace("", "").replace("Final Answer:", "").strip()
#         text = REASONING_TEMPLATE.format(
#             system_prompt=REASONING_ON_SYSTEM, question=q,
#             chain_of_thought=cot, final_answer=final_answer)
#     else:
#         text = NON_REASONING_TEMPLATE.format(
#             system_prompt=REASONING_OFF_SYSTEM, question=q,
#             final_answer=a)

#     return {"text": text, "is_valid": True}

## Step 3: Load and Format the Custom SFT Training Dataset

- **Custom SFT Data:** Contains verified mathematical, cipher, bit-manipulation, and physics logic paths. Guessed or unverified Chain-of-Thought (CoT) paths have been purged.

The CSV file is loaded directly from the Kaggle input directory. The unique columns (`prompt` and `cot`) are normalized to a standard `question` / `answer` format, shuffled, and formatted into strict ChatML syntax. Samples that are invalid, too short, or exceed the maximum sequence length are automatically discarded.

In [ ]:
# import datasets
# from datasets import load_dataset, Dataset
# import gc

# # Prevent huggingface from logging thousands of progress bar lines, which crashes papermill
# datasets.utils.logging.disable_progress_bar()

# print("\n Loading Custom Verified SFT Dataset ...")

# # ── Custom SFT Dataset path ──
# SFT_DATA_PATH = "/kaggle/input/datasets/banwait13/lora-adapter/DATASETS/SFT_FLAWLESS_FINAL.csv"

# def load_custom_dataset(path):
#     try:
#         print(f"    Loading verified logic paths from {path}...")
#         # Since the verified dataset is highly curated, we load it directly into memory
#         ds = load_dataset("csv", data_files=path, split="train")
#         print(f"    Materialized {len(ds):,} rows | Columns: {ds.column_names}")
#         return ds
#     except Exception as e:
#         print(f"    FAILED to load custom dataset — {e}")
#         raise

# # 1. Load the data
# dataset = load_custom_dataset(SFT_DATA_PATH)

# # We now catch 'instruction' and 'response' natively
# def normalize_to_qa(sample):
#     question = sample.get("instruction", sample.get("prompt", sample.get("question", "")))
#     answer = sample.get("response", sample.get("cot", sample.get("answer", "")))
#     return {"question": question, "answer": answer}

# print("   Normalizing columns...")
# dataset = dataset.map(normalize_to_qa, num_proc=2, desc="Normalizing to QA")

# # 3. Shuffle the logic puzzles so the AI doesn't memorize patterns
# dataset = dataset.shuffle(seed=3407)
# print(f"\n   Raw samples loaded: {len(dataset):,}")

# # 4. Apply universal formatting (Executes your ChatML structure)
# print("   Applying formatting + quality filter...")
# dataset = dataset.map(format_sample, num_proc=2, desc="Formatting")

# if "is_valid" in dataset.column_names:
#     dataset = dataset.filter(lambda x: x["is_valid"])

# # 5. Fast length filter (char-based heuristic: ~4 chars per token)
# MAX_CHAR_LEN = MAX_SEQ_LEN * 4  # ~16384 chars ≈ 4096 tokens at the model max sequence length
# dataset = dataset.filter(lambda x: 0 < len(x["text"]) <= MAX_CHAR_LEN)

# # 6. Clean up columns for SFTTrainer
# keep_cols = {"text"}
# drop_cols = [c for c in dataset.column_names if c not in keep_cols]
# if drop_cols:
#     dataset = dataset.remove_columns(drop_cols)

# # Free up system memory
# gc.collect()

# print(f"\n    Final SURVIVING training samples: {len(dataset):,}")
# if len(dataset) > 0:
#     print("\n Sample Preview (first 500 chars):")
#     print(dataset[0]["text"][:500] + "...\n")
# else:
#     print("\n  ALL SAMPLES DISCARDED! Check column names or data format.\n")

## Step 4: Tokenize the Dataset

The formatted text samples are tokenized using the Nemotron tokenizer. Sequences longer than `MAX_SEQ_LEN` (4096 tokens) are truncated. The tokenized dataset is then split 95/5 into a training set and a validation set. The validation set is used during training to monitor for overfitting.

In [ ]:
# print(" Tokenizing dataset...")

# def tokenize_fn(examples):
#     """Tokenize text for causal LM training."""
#     return tokenizer(
#         examples["text"],
#         truncation=True,
#         max_length=MAX_SEQ_LEN,
#         padding=False,
#     )

# tokenized_dataset = dataset.map(
#     tokenize_fn,
#     batched=True,
#     remove_columns=dataset.column_names,
#     num_proc=2,
#     desc="Tokenizing",
# )

# # Free the raw text dataset from memory
# del dataset
# gc.collect()

# # Split off 5% of the tokenized dataset for validation
# from datasets import DatasetDict
# split = tokenized_dataset.train_test_split(test_size=0.05, seed=3407)
# train_dataset = split["train"]
# eval_dataset  = split["test"]
# del tokenized_dataset
# gc.collect()

# print(f"    Train samples: {len(train_dataset):,}")
# print(f"    Eval samples:  {len(eval_dataset):,}")
# if len(train_dataset) > 0:
#     sample_len = len(train_dataset[0]["input_ids"])
#     print(f"   First sample length: {sample_len} tokens")

## Step 5: Configure the Trainer

The HuggingFace `Trainer` is configured with the following key settings:

- **Completion-only loss (TRL):** The loss is computed only on the assistant's response, not the prompt. This is achieved by masking all tokens before the `<|im_start|>assistant` token.
- **Effective batch size of 8:** Achieved via 1 sample per device with 8 gradient accumulation steps, keeping system RAM usage within the 16 GB Kaggle limit.
- **BFloat16 precision:** Matches the model's native dtype. No additional quantization is applied during training.
- **Paged AdamW 8-bit optimizer:** Offloads optimizer states to CPU pages, significantly reducing VRAM compared to a standard Adam optimizer.
- **Cosine learning rate schedule:** Learning rate decays smoothly from the peak to near zero over the full epoch.
- **Evaluation every 250 steps:** Both `per_device_eval_batch_size` and `eval_accumulation_steps` are set to 1 to prevent out-of-memory errors at eval time, as HuggingFace defaults these to higher values.

In [ ]:
# from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
# import torch

# try:
#     from trl import DataCollatorForCompletionOnlyLM
#     response_template = "<|im_start|>assistant\n"
#     data_collator = DataCollatorForCompletionOnlyLM(
#         response_template=response_template,
#         tokenizer=tokenizer
#     )
#     print(" Activated TRL Completion-Only Loss (Prompt Masked).")
# except Exception as e:
#     data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
#     print(f" Falling back to standard LM collator: {e}")


# BATCH_SIZE        = 8       # ABSOLUTE MAX: Packs the 40GB VRAM completely full
# GRAD_ACCUMULATION = 2       # 8 (Batch) x 2 (Accum) = 16 Effective Batch
# NUM_EPOCHS        = 1       # Reads the entire dataset from start to finish once
# WARMUP_RATIO      = 0.10    
# LEARNING_RATE     = 2e-4    
# SFT_OUTPUT_DIR    = "/kaggle/working/nemotron_sft_output"

# training_args = TrainingArguments(
#     output_dir                   = SFT_OUTPUT_DIR,
#     per_device_train_batch_size  = BATCH_SIZE,
#     gradient_accumulation_steps  = GRAD_ACCUMULATION,
#     gradient_checkpointing       = True,
#     gradient_checkpointing_kwargs= {"use_reentrant": False},
#     optim                        = "paged_adamw_8bit",

#     # ── MAXIMUM EXPOSURE ─────────────────────────────────────────
#     # -1 tells the trainer to ignore step limits and run the full epoch.
#     max_steps                    = -1,
#     num_train_epochs             = NUM_EPOCHS,   

#     # Check results and save only at the very end to save processing time
#     eval_strategy                = "epoch",      
#     save_strategy                = "epoch",      
#     per_device_eval_batch_size   = BATCH_SIZE,   
#     eval_accumulation_steps      = 1,

#     learning_rate                = LEARNING_RATE,
#     warmup_ratio                 = WARMUP_RATIO,
#     lr_scheduler_type            = "cosine",
#     weight_decay                 = 0.01,
#     max_grad_norm                = 1.0,

#     bf16                         = True,         
#     fp16                         = False,

#     logging_steps                = 10,
#     seed                         = 3407,
#     report_to                    = "none",
#     disable_tqdm                 = False,
#     dataloader_pin_memory        = True,
#     dataloader_num_workers       = 2,            
#     remove_unused_columns        = False,
# )

# trainer = Trainer(
#     model         = model,
#     args          = training_args,
#     train_dataset = train_dataset,
#     eval_dataset  = eval_dataset,
#     data_collator = data_collator,
# )

# print(f"\n SFT Trainer armed.")
# print(f"   Max steps    : {training_args.max_steps}")
# print(f"   Effective BS : {BATCH_SIZE * GRAD_ACCUMULATION}")
# print(f"   LR           : {LEARNING_RATE}")
# print(f"   Max seq len  : {MAX_SEQ_LEN}")


## Step 6: VRAM Check Before Training

Before launching the training run, available GPU memory is measured. If free VRAM falls below 4 GB, a warning is printed. In that case, reduce `MAX_SEQ_LEN` or the LoRA rank before proceeding.

In [ ]:
# gc.collect()
# torch.cuda.empty_cache()

# # 1. Fetch the GPU Hardware Stats
# gpu_stats = torch.cuda.get_device_properties(0)
# start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
# max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
# free_memory = max_memory - start_gpu_memory

# print(f"\n GPU: {gpu_stats.name}")
# print(f"   Total VRAM:    {max_memory} GB")
# print(f"   Reserved VRAM: {start_gpu_memory} GB")
# print(f"   Free VRAM:     {free_memory:.2f} GB\n")

# is_critical = free_memory < 4.0
# if is_critical:
#     print(" WARNING: Less than 4GB free VRAM. Training may OOM.")
#     print("   Consider reducing MAX_SEQ_LEN or LoRA rank.")

# # ==========================================
# # 2. SEABORN VRAM HUD GRAPH
# # ==========================================
# sns.set_theme(style="dark", rc={
#     "axes.facecolor": "#121212", 
#     "figure.facecolor": "#121212",
#     "text.color": "white"
# })

# # Create a DataFrame for Seaborn to digest
# df = pd.DataFrame({
#     'Component': ['VRAM'],
#     'Used': [start_gpu_memory],
#     'Total': [max_memory]
# })

# f, ax = plot.subplots(figsize=(8, 1.2))

# # Plot the background bar (Total VRAM / Free Space) in dark gray
# sns.barplot(x="Total", y="Component", data=df, color="#2a2a2a", ax=ax)

# # Plot the foreground bar (Used VRAM) overriding the color based on danger levels
# color_reserved = '#ff3333' if is_critical else '#00e5ff' # Red if critical, Cyan if safe
# sns.barplot(x="Used", y="Component", data=df, color=color_reserved, ax=ax)

# # Clean up the axes for the pure HUD look
# ax.set(ylabel="", xlabel="")
# ax.set_xticks([])
# ax.set_yticks([])
# sns.despine(left=True, bottom=True)

# # Add dynamic text overlays right inside the bars
# if start_gpu_memory > 2.0:
#     plot.text(start_gpu_memory / 2, 0, f"USED: {start_gpu_memory:.1f} GB", 
#              color='#121212', va='center', ha='center', fontweight='bold', fontsize=10)
    
# if free_memory > 2.0:
#     plot.text(start_gpu_memory + (free_memory / 2), 0, f"FREE: {free_memory:.1f} GB", 
#              color='#888888', va='center', ha='center', fontweight='bold', fontsize=10)

# # THE FIX: Removed letterspacing, manually spaced the title for the cinematic look
# plot.title("S Y S T E M   M E M O R Y   S T A T U S", color='white', fontsize=11, pad=10, fontweight='bold')
# plot.tight_layout()
# plot.show()

## Step 7: Run Phase 1 Training

The trainer is launched here. A post-training VRAM report is printed showing peak memory usage, total training time, and final training loss.

In [ ]:
# print("\n COMMENCING PHASE 1 TRAINING...\n")
# trainer_stats = trainer.train()

# # --- Post-training VRAM report ---
# used_memory     = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
# used_memory_lora= round(used_memory - start_gpu_memory, 3)
# pct_memory      = round(used_memory / max_memory * 100, 3)

# print(f"\n TRAINING COMPLETE.")
# print(f"   Peak VRAM used:     {used_memory} GB ({pct_memory}% of total)")
# print(f"   VRAM for training:  {used_memory_lora} GB")
# print(f"   Training time:      {trainer_stats.metrics['train_runtime']:.2f} seconds")
# print(f"   Training loss:      {trainer_stats.metrics['train_loss']:.4f}")

## Step 8: Save the LoRA Adapter

The trained LoRA adapter weights and the tokenizer are saved to disk. Only the adapter weights are saved, not the full base model, keeping the output size manageable. Adapter size and remaining disk space are reported after saving.

In [ ]:
# ADAPTER_SAVE_PATH = "/kaggle/working/nemotron_lora_adapter"
# model.save_pretrained(ADAPTER_SAVE_PATH, safe_serialization=True)
# tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

# # Report disk usage
# import shutil
# adapter_size = sum(
#     os.path.getsize(os.path.join(dp, f))
#     for dp, _, fns in os.walk(ADAPTER_SAVE_PATH)
#     for f in fns
# ) / (1024 * 1024)
# disk_usage = shutil.disk_usage("/kaggle/working")
# print(f"\n LoRA adapter saved to: {ADAPTER_SAVE_PATH}")
# print(f"   Adapter size: {adapter_size:.1f} MB")
# print(f"   Disk remaining: {disk_usage.free / (1024**3):.1f} GB")

## Step 9: Inference Test

A quick inference test is run to verify the model has learned the required output format. A math problem is passed to the model with the reasoning system prompt active. The generated output is checked for:

- A closing `</think>` tag -- confirms the model learned to close its chain-of-thought block.
- A `\\boxed{}` final answer -- confirms the model learned to produce a structured answer.

If both are present, Phase 1 is considered successful.

In [ ]:
# print("\n RUNNING INFERENCE TEST...\n")

# model.eval()

# # Build prompt directly — model generates everything after <think>
# test_prompt = (
#     "<|im_start|>system\n"
#     "detailed thinking on<|im_end|>\n"
#     "<|im_start|>user\n"
#     "If f(x) = x^2 + 3x - 4, find all values of x where f(x) = 0. Show full working.<|im_end|>\n"
#     "<|im_start|>assistant\n"
#     "<think>\n"
# )

# inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
# prompt_len = inputs["input_ids"].shape[1]

# with torch.inference_mode():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens  = 1024,
#         temperature     = 0.6,
#         top_p           = 0.95,
#         do_sample       = True,
#         eos_token_id    = tokenizer.eos_token_id,
#         repetition_penalty = 1.1
#     )

# # Decode full output and generated-only portion separately
# full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
# generated_text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=False)

# print("--- FULL OUTPUT ---")
# print(full_response)
# print("-------------------")
# print("\n--- GENERATED PORTION ONLY ---")
# print(generated_text)
# print("-------------------")

# # Validate format on generated text only (not the injected prompt)
# has_think_close = "</think>" in generated_text
# has_boxed = "\\boxed{" in generated_text

# if has_think_close and has_boxed:
#     print("\n PHASE 1 MISSION SUCCESS — <think>...</think> + \\boxed{} confirmed.")
# elif has_think_close:
#     print("\n  </think> found but \\boxed{} missing in output.")
# elif has_boxed:
#     print("\n  \\boxed{} found but </think> closure missing — format partially learned.")
# else:
#     print("\n WARNING — Neither </think> nor \\boxed{} detected. More training may be needed.")

# # Final cleanup
# del inputs, outputs
# gc.collect()
# torch.cuda.empty_cache()

# print("PHASE 1 COMPLETE (supervised Fine tuning for Nemotron completed).")

# Section 2: Phase 2—Custom Rl Architecture inspired from BAPO (Balanced Policy Optimization with Adaptive Clipping) & DAPO (Decoupled Clip and Dynamic sAmpling Policy Optimization)

Standard GRPO has four structural flaws when training reasoning models on hard mathematical datasets:
1. **Symmetric Clipping:** Caps the learning signal on rare, hard-won correct answers.
2. **All-Miss Groups:** Generates zero gradients but still wastes massive backward-pass compute.
3. **Sequence-Level Loss:** Disincentivizes long, complex reasoning chains.
4. **The Verbosity Trap:** Token-level loss can cause the model to artificially bloat its output to farm gradients.

**The Solution:** I am implementing **BAPO** (Balanced Policy Optimization with Adaptive Clipping) & DAPO (Decoupled Clip and Dynamic Sampling Policy Optimization). We introduce asymmetric clipping (eps_low=0.2, eps_high=0.5) to maximize learning on breakthroughs, sampling to skip dead batches, token-level loss weighting, and a length-penalized reward function to enforce efficient logic.

**NOTE:** I am implemented DAPO first but with more research found that I need to combine the architectures of the two, to get the optimal solution

>  PPO Clip Objective Function (schulmen et al)
> $$L^{CLIP}(\theta) = \hat{\mathbb{E}}_{t} \left[ \min \left( r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t \right) \right]$$

> GRPO Objective Function (shao et al 2024)
> $$\mathcal{J}_{GRPO}(\theta) = \mathbb{E} \left[ \frac{1}{G} \sum_{i=1}^G \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \left\{ \min \left( r_{i,t}(\theta) \hat{A}_{i,t}, \text{clip}(r_{i,t}(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_{i,t} \right) - \beta \mathbb{D}_{KL} [\pi_\theta \mid\mid \pi_{\text{ref}}] \right\} \right]$$

> DAPO Objective Function(yu et al 2025)
> $$\mathcal{J}_{DAPO}(\theta) = \mathbb{E} \left[ \frac{1}{\sum_{i=1}^G |o_i|} \sum_{i=1}^G \sum_{t=1}^{|o_i|} \min \left( r_{i,t}(\theta)\hat{A}_{i,t}, \text{clip}(r_{i,t}(\theta), 1 - \epsilon_{low}, 1 + \epsilon_{high})\hat{A}_{i,t} \right) \right]$$
>
> $$s.t.\ \ 0 < |\{o_i \mid \text{is\_equivalent}(a, o_i)\}| < G$$

> BAPO Objective Function(xi et al 2025)
> $$\mathcal{J}_{BAPO}(\theta) = \mathbb{E} \left[ \frac{1}{G} \sum_{i=1}^G \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \min \left( r_{i,t}(\theta) \hat{A}_{i,t}, \text{clip}(r_{i,t}(\theta), c_{low}^{*}, c_{high}^{*}) \hat{A}_{i,t} \right) \right]$$
> with bounds to be calculated as:
> 
> $$\frac{\left| \sum_{A_t > 0} [\min(r_t A_t, \text{clip}(r_t, 0, c_{high}^{*}) A_t)] \right|}{\left| \sum_{A_t < 0} [\min(r_t A_t, \text{clip}(r_t, c_{low}^{*}, c_{high}^{*}) A_t)] \right|} \ge \rho_0$$

### OUR ARCHITECTURE:
### We propose a custom architecture specifically built for the Kaggle environment, considering the normalizer from DAPO that is $$ \frac{1}{G} \sum_{i=1}^G \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} $$ and bounds from BAPO  $$c_{low}^{*}, c_{high}^{*}$$ Combining these two, we get the following objective function: 
> Custom Kaggle Hybrid Objective Function
> $$\mathcal{J}_{BANWAIT}(\theta) = \mathbb{E} \left[ \frac{1}{\sum_{i=1}^G |o_i|} \sum_{i=1}^G \sum_{t=1}^{|o_i|} \min \left( r_{i,t}(\theta)\hat{A}_{i,t}, \text{clip}(r_{i,t}(\theta), c_{low}^{*}, c_{high}^{*})\hat{A}_{i,t} \right) \right]$$
>
> $$s.t.\ \ 0 < |\{o_i \mid \text{is\_equivalent}(a, o_i)\}| < G$$
> 
> > WITH BOUNDS
> > $$\frac{\left| \sum_{A_t > 0} [\min(r_t A_t, \text{clip}(r_t, 0, c_{high}^{*}) A_t)] \right|}{\left| \sum_{A_t < 0} [\min(r_t A_t, \text{clip}(r_t, c_{low}^{*}, c_{high}^{*}) A_t)] \right|} \ge \rho_0$$

Because we are utilizing Parameter-Efficient Fine-Tuning (LoRA), the base model is permanently frozen. The "Reference Model" ($\pi_{ref}$) is mathematically defined as the exact state where all trainable LoRA matrices ($\Delta W$) equal $0$.

Using a second-order Taylor expansion and assuming Lipschitz continuity of the neural network, we mathematically prove that the output-space KL Divergence is strictly bounded by the physical size of the LoRA weights. We can entirely replace the massive $\mathcal{O}(V)$ vocabulary probability calculation with an ultra-light $\mathcal{O}(1)$ parameter-space tether: **The Squared Frobenius Norm ($L_2$ Norm)**.

### Final Custom Objective Function:
$$\mathcal{J}_{Banwait}(\theta) = \mathbb{E}_{q \sim \mathcal{D}, \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}} \left[ \frac{1}{\sum_{i=1}^G |o_i|} \sum_{i=1}^G \sum_{t=1}^{|o_i|} \min \left( r_{i,t}(\theta)\hat{A}_{i,t}, \text{clip}(r_{i,t}(\theta), c_{low}^{*}, c_{high}^{*})\hat{A}_{i,t} \right) \right] - \frac{\beta}{2} \sum_{m \in \mathcal{M}_{LoRA}} ||\Delta W_m||_F^2$$

### ARCHIETECTURE BREAKDOWN 
       [ 1. PPO (Baseline) ] ────► Symmetric clipping, high-VRAM dual-model dependency (Policy + Critic)
                 │
                 ▼
       [ 2. GRPO (VRAM Saver) ] ──► Deletes Critic model, introduces Group-Relative Advantage & KL band
                 │
                 ▼
       [ 3. DAPO (Length Normal) ] ► Introduces Global Length Normalizer & Asymmetric bounds
                 │
                 ▼
       [ 4. BAPO (Dynamic Suspen) ] ► Discards rigid limits for Dynamic Bounds preserving entropy
                 │
                 ▼
      [ 5. Custom Kaggle Architecture ] ► Hybrid: DAPO Global Normalizer + BAPO Dynamic Bounds
                                      Data Filter active with custom Kl divergence for efficiency

### INSPIRATION(GITHUB) 
Code from TRL's github: https://github.com/huggingface/trl
Code from BAPO's github: https://github.com/WooooDyy/BAPO

### CITATIONS

**TRL**

@software{vonwerra2020trl,
  title   = {{TRL: Transformers Reinforcement Learning}},
  author  = {von Werra, Leandro and Belkada, Younes and Tunstall, Lewis and Beeching, Edward and Thrush, Tristan and Lambert, Nathan and Huang, Shengyi and Rasul, Kashif and Gallouédec, Quentin},
  license = {Apache-2.0},
  url     = {https://github.com/huggingface/trl},
  year    = {2020}
}

**GRPO**

@misc{shao2024deepseekmathpushinglimitsmathematical,
      title={DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models}, 
      author={Zhihong Shao and Peiyi Wang and Qihao Zhu and Runxin Xu and Junxiao Song and Xiao Bi and Haowei Zhang and Mingchuan Zhang and Y. K. Li and Y. Wu and Daya Guo},
      year={2024},
      eprint={2402.03300},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2402.03300}, 
}

**BAPO**

@misc{xi2025bapostabilizingoffpolicyreinforcement,
      title={BAPO: Stabilizing Off-Policy Reinforcement Learning for LLMs via Balanced Policy Optimization with Adaptive Clipping}, 
      author={Zhiheng Xi and Xin Guo and Yang Nan and Enyu Zhou and Junrui Shen and Wenxiang Chen and Jiaqi Liu and Jixuan Huang and Zhihao Zhang and Honglin Guo and Xun Deng and Zhikai Lei and Miao Zheng and Guoteng Wang and Shuo Zhang and Peng Sun and Rui Zheng and Hang Yan and Tao Gui and Qi Zhang and Xuanjing Huang},
      year={2025},
      eprint={2510.18927},
      archivePrefix={arXiv},
      primaryClass={cs.LG},
      url={https://arxiv.org/abs/2510.18927}, 
}

**DAPO**

@misc{yu2025dapoopensourcellmreinforcement,
      title={DAPO: An Open-Source LLM Reinforcement Learning System at Scale}, 
      author={Qiying Yu and Zheng Zhang and Ruofei Zhu and Yufeng Yuan and Xiaochen Zuo and Yu Yue and Weinan Dai and Tiantian Fan and Gaohong Liu and Lingjun Liu and Xin Liu and Haibin Lin and Zhiqi Lin and Bole Ma and Guangming Sheng and Yuxuan Tong and Chi Zhang and Mofan Zhang and Wang Zhang and Hang Zhu and Jinhua Zhu and Jiaze Chen and Jiangjie Chen and Chengyi Wang and Hongli Yu and Yuxuan Song and Xiangpeng Wei and Hao Zhou and Jingjing Liu and Wei-Ying Ma and Ya-Qin Zhang and Lin Yan and Mu Qiao and Yonghui Wu and Mingxuan Wang},
      year={2025},
      eprint={2503.14476},
      archivePrefix={arXiv},
      primaryClass={cs.LG},
      url={https://arxiv.org/abs/2503.14476}, 
}

## Step 1: Reload the SFT Adapter and Load the GRPO Dataset

The base model is already in memory from Section 1. We reload the SFT adapter so GRPO
starts from the fine-tuned checkpoint — not from the raw base model.

The GRPO dataset (`FINAL_DATA_GRPO.csv`) must contain **`question`** and **`answer`** columns.
The `answer` column must include a `\\boxed{}` ground-truth so the correctness reward can fire.

In [ ]:
import gc
import torch
import os
import re
import math
import pandas as pd
from datasets import load_dataset, Dataset
from peft import PeftModel, LoraConfig, get_peft_model
import peft.import_utils
import peft.tuners.lora.torchao
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False

from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
import collections
datasets.utils.logging.disable_progress_bar()

import functools as _ft
from torch.utils.checkpoint import checkpoint as _torch_ckpt

BASE_MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
ADAPTER_3_PATH = "/kaggle/input/datasets/banwait13/lora-adapter/NEMOTRON/1"

if 'model' not in globals() or model is None:
    print("[DAPO] Loading Base Model...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
    if tokenizer.unk_token is not None:
        tokenizer.pad_token = tokenizer.unk_token
    else:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print("[DAPO] Attaching Adapter 3 as trainable...")
    model = PeftModel.from_pretrained(model, ADAPTER_3_PATH, is_trainable=True)
    print(f"[DAPO] Adapter loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
else:
    print("[DAPO] Model already in memory.")

# ── Freeze MoE expert LoRA weights + lm_head ─────────────────────────────────
# Only train Attention (q/k/v/o_proj) and Mamba (in/out_proj) LoRA parameters.
# MoE expert projections (up_proj, down_proj) are frozen to reduce VRAM and
# prevent the RL gradient from corrupting the MoE routing learned during SFT.
_BAPO_FROZEN_MODULES = {"lm_head", "up_proj", "down_proj"}
_frozen_count = 0
for name, param in model.named_parameters():
    if any(mod in name for mod in _BAPO_FROZEN_MODULES):
        param.requires_grad = False
        _frozen_count += 1
_trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[BAPO] Froze {_frozen_count} tensors (lm_head + MoE up/down_proj).")
print(f"[BAPO] Active LoRA targets: q_proj, k_proj, v_proj, o_proj, in_proj, out_proj")
print(f"[BAPO] Trainable params after freeze: {_trainable_after:,}")

model.config.use_cache = False
# FIX: use_reentrant=True is required for Mamba's custom CUDA autograd functions.
# use_reentrant=False causes cudaErrorInvalidConfiguration in causal_conv1d_bwd.
_gc_fn = _ft.partial(_torch_ckpt, use_reentrant=True)
for _m in model.modules():
    if hasattr(_m, 'gradient_checkpointing'):
        _m.gradient_checkpointing = True
    _m._gradient_checkpointing_func = _gc_fn

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

print("[DAPO] Gradient checkpointing force-applied (use_reentrant=True).")

# ── CLEANUP: Remove any stale monkeypatches from previous Jupyter runs ─────────
_base = getattr(model, "base_model", model)
if "forward" in _base.__dict__:
    del _base.__dict__["forward"]
    print("[BAPO] Cleaned stale forward monkeypatch from previous run.")
if "prepare_inputs_for_generation" in model.__dict__:
    del model.__dict__["prepare_inputs_for_generation"]
    print("[BAPO] Cleaned stale prepare_inputs monkeypatch from previous run.")


# ── Load FULL dataset (1000 samples) ─────────────────────────────────────────
_raw_dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/banwait13/lora-adapter/DATASETS/GRPO_TEST.csv",
    split="train",
)

def normalize_cols(sample):
    question = (
        sample.get("instruction") or sample.get("prompt")
        or sample.get("question") or ""
    )
    answer = (
        sample.get("response") or sample.get("answer")
        or sample.get("output") or ""
    )
    return {
        "question":    str(question).strip(),
        "answer":      str(answer).strip(),
    }

_raw_dataset = _raw_dataset.map(normalize_cols, num_proc=2)

# ── Auto-categorize puzzles from instruction text ─────────────────────────────
def _categorize_instruction(text: str) -> str:
    t = str(text).lower()
    if "gravitational" in t or "falling distance" in t or "d = 0.5*g*t" in t:
        return "gravity"
    elif "bit manipulation" in t or "binary" in t:
        return "bit_manipulation"
    elif "decrypt" in t or "encrypt" in t or "cipher" in t:
        return "encryption"
    elif "unit conversion" in t or "convert the following measurement" in t:
        return "unit_conversion"
    elif "numeral" in t or "roman" in t:
        return "roman_numeral"
    elif "transformation rules" in t or "transformation" in t:
        return "transformation"
    else:
        return "other"

_raw_dataset = _raw_dataset.map(
    lambda x: {"puzzle_type": _categorize_instruction(x["question"])},
    desc="Categorizing puzzles",
)

# ── Build prompts ─────────────────────────────────────────────────────────────
def build_grpo_prompt(example: dict) -> str:
    question_with_hint = (
        example["question"]
        + "\nThink step by step inside <think> tags, then give your final answer inside \\boxed{}."
        + "\nFormat required: <think>your reasoning</think> \\boxed{your answer}"
    )
    try:
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": question_with_hint}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        prompt = (
            "<|im_start|>system\ndetailed thinking on<|im_end|>\n"
            f"<|im_start|>user\n{question_with_hint}<|im_end|>\n"
            "<|im_start|>assistant\n<think>\n"   # ← force open the think block
        )
    return prompt

_raw_dataset = _raw_dataset.map(
    lambda x: {"prompt": build_grpo_prompt(x)},
    desc="Building DAPO prompts",
)

# ── Per-category dataset splits for sequential DAPO runs ──────────────────────
# Strategy: 3 sequential runs on the SAME adapter, each specializing in
# a cluster of related problem types. This prevents the model from having
# to context-switch between completely unrelated reasoning strategies.
#
# Run ordering: easiest → hardest (so the adapter warms up on solvable problems)

BAPO_RUNS = [
    {
        "name": "full_training",
        "cats": ["gravity", "unit_conversion", "roman_numeral",
                 "transformation", "encryption", "bit_manipulation", "other"],
        "description": "Single continuous BAPO run on all puzzle types",
        "lr": 5e-7,
        "max_steps": 200,
    }
]


# Build category subsets (shuffle within each, but keep category grouping)
bapo_run_datasets = {}
for run_cfg in BAPO_RUNS:
    subset = _raw_dataset.filter(
        lambda x, cats=run_cfg["cats"]: x["puzzle_type"] in cats,
        desc=f"Filtering {run_cfg['name']}",
    )
    # Shuffle within the subset
    subset = subset.shuffle(seed=3407)
    bapo_run_datasets[run_cfg["name"]] = subset

# Clean up columns: keep only what BAPOTrainer needs
keep = {"question", "answer", "puzzle_type", "prompt"}
for name in bapo_run_datasets:
    ds = bapo_run_datasets[name]
    drop = [c for c in ds.column_names if c not in keep]
    if drop:
        bapo_run_datasets[name] = ds.remove_columns(drop)

# Also keep a combined dataset as fallback
grpo_dataset = _raw_dataset.shuffle(seed=3407)
drop = [c for c in grpo_dataset.column_names if c not in keep]
if drop:
    grpo_dataset = grpo_dataset.remove_columns(drop)

# ── Report ────────────────────────────────────────────────────────────────────
print(f"\n[DAPO] Full dataset: {len(_raw_dataset):,} samples")
counts = collections.Counter(_raw_dataset["puzzle_type"])
for k, v in sorted(counts.items()):
    print(f"  {k:20s}: {v:4d}  ({100*v/len(_raw_dataset):.1f}%)")

print(f"\n[DAPO] Sequential run plan ({len(BAPO_RUNS)} runs):")
for run_cfg in BAPO_RUNS:
    ds = bapo_run_datasets[run_cfg["name"]]
    print(f"  Run '{run_cfg['name']}': {len(ds):4d} samples | LR={run_cfg['lr']:.0e} | steps={run_cfg['max_steps']} | {run_cfg['description']}")

print(f"\n[DAPO] Total training steps: {sum(r['max_steps'] for r in BAPO_RUNS)}")
gc.collect()

## Step 2: Define the BAPO Reward Functions

BAPO scores completions automatically using **reward functions**. We add a new **Verbosity Penalty** here (Flaw 4 Fix) to ensure the model finds the most mathematically efficient reasoning path rather than just rambling to farm gradients.

In [ ]:
BOXED_PATTERN = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

# Completion length budget — must match max_completion_length in BAPOConfig
MAX_COMPLETION_LENGTH = 2048
LCACHE = 100   # Soft punishment window: last 100 tokens before hard truncation

def _get_text(comp):
    return comp[0]["content"] if isinstance(comp, list) else comp

def extract_boxed(text: str) -> str:
    matches = BOXED_PATTERN.findall(text)
    return matches[-1].strip() if matches else ""

def _norm(s: str) -> str:
    return re.sub(r"\s+", "", s.strip()).lower()

def _try_float(s: str):
    s = s.strip()
    s = re.sub(r"\\text\{[^}]*\}", "", s)
    s = re.sub(r"\\,|\\;", "", s)
    s = s.replace(",", "").replace("%", "")
    frac = re.match(r"\\frac\{([^}]+)\}\{([^}]+)\}", s)
    if frac:
        try:
            return float(frac.group(1)) / float(frac.group(2))
        except Exception:
            pass
    try:
        return float(s)
    except Exception:
        return None


# ── Reward 1: FORMAT ─────────────────────────────────────────────────────────
def reward_format(completions, **kwargs) -> list:
    scores = []
    for comp in completions:
        text      = _get_text(comp)
        has_close = "</think>" in text

        if not has_close:
            scores.append(-2.0)  # UNIVERSAL FLOOR 1: Massive penalty
        else:
            tail = text.split("</think>")[-1]
            if BOXED_PATTERN.search(tail):
                scores.append(1.0)   # Perfect
            else:
                scores.append(-1.0)  # Compound penalty for missing box
    return scores


# ── Reward 2: THINKING QUALITY ───────────────────────────────────────────────
def reward_thinking_quality(completions, **kwargs) -> list:
    scores = []
    for comp in completions:
        text = _get_text(comp)

        if "</think>" not in text:
            # DO NOT RETURN 0.0 HERE. Compound the floor penalty.
            scores.append(-2.0) # UNIVERSAL FLOOR 2
            continue

        if "<think>" in text:
            body = text.split("<think>")[-1].split("</think>")[0].strip()
        else:
            body = text.split("</think>")[0].strip()

        approx_tokens = len(body) / 4

        if approx_tokens >= 80:
            scores.append(1.5)    
        elif approx_tokens >= 30:
            scores.append(0.5)   
        else:
            scores.append(-1.0)  
    return scores


# ── Reward 3: CORRECTNESS ─────────────────────────────────────────────────────
def reward_correctness(completions, answer, **kwargs) -> list:
    scores = []
    for comp, gt in zip(completions, answer):
        text     = _get_text(comp)
        gt_clean = extract_boxed(gt) if BOXED_PATTERN.search(gt) else gt.strip()
        pred     = extract_boxed(text)

        if not pred:
            # DO NOT RETURN 0.0 HERE. Compound the floor penalty.
            scores.append(-2.0) # UNIVERSAL FLOOR 3
        elif _norm(pred) == _norm(gt_clean):
            scores.append(4.0)    
        else:
            scores.append(0.0)   # NEUTRAL: Trying and failing is acceptable.
    return scores

# ── Reward 4: NUMERICAL EQUIVALENCE ──────────────────────────────────────────
# Unchanged — handles physics, unit conversion, algebra.
# 1% relative tolerance for full credit; 5% for partial.
def reward_numerical(completions, answer, **kwargs) -> list:
    scores = []
    for comp, gt in zip(completions, answer):
        text     = _get_text(comp)
        gt_clean = extract_boxed(gt) if BOXED_PATTERN.search(gt) else gt.strip()
        pred     = extract_boxed(text)

        if not pred:
            scores.append(0.0)
            continue

        gt_val, pred_val = _try_float(gt_clean), _try_float(pred)
        if gt_val is None or pred_val is None:
            scores.append(0.0)
            continue

        if math.isclose(gt_val, pred_val, rel_tol=0.01, abs_tol=1e-6):
            scores.append(3.0)
        elif math.isclose(gt_val, pred_val, rel_tol=0.05, abs_tol=1e-4):
            scores.append(1.0)
        else:
            scores.append(0.0)
    return scores


# ── Reward 5: ANTI-HACKING ────────────────────────────────────────────────────
# Unchanged — detects GT pasted in think block with wrong \boxed{}.
def reward_no_hacking(completions, answer, **kwargs) -> list:
    scores = []
    for comp, gt in zip(completions, answer):
        text     = _get_text(comp)
        gt_clean = extract_boxed(gt) if BOXED_PATTERN.search(gt) else gt.strip()
        pred     = extract_boxed(text)
        score    = 0.0

        if "</think>" in text:
            if "<think>" in text:
                think_body = text.split("<think>")[-1].split("</think>")[0]
            else:
                think_body = text.split("</think>")[0]

            gt_in_think  = _norm(gt_clean) in _norm(think_body)
            pred_correct = _norm(pred) == _norm(gt_clean)
            if gt_in_think and not pred_correct:
                score = -0.5
        scores.append(score)
    return scores


# ── Reward 6: OVERLONG SHAPING ────────────────────────────────────────────────
# CHANGE: Updated MAX_COMPLETION_LENGTH to match BAPOConfig (768, not 2048).
# The previous mismatch meant most completions were never penalised.
def reward_overlong_shaping(completions, **kwargs) -> list:
    scores = []
    for comp in completions:
        text       = _get_text(comp)
        approx_len = len(text) / 4   # ~4 chars per token

        if approx_len <= (MAX_COMPLETION_LENGTH - LCACHE):
            scores.append(0.0)
        elif approx_len < MAX_COMPLETION_LENGTH:
            penalty = ((MAX_COMPLETION_LENGTH - LCACHE) - approx_len) / LCACHE
            scores.append(penalty)   # Negative, linear ramp
        else:
            scores.append(-1.0)
    return scores


print("[BAPO] 6-Reward system armed. Reward landscape:")
print()
print("  Scenario                      Weighted Total")
print("  ──────────────────────────────────────────────")
print("  no think + not truncated      :  -2.0  (floor)")
print("  think + no boxed              :  -0.75 (format partial + thin thinking)")
print("  think + wrong boxed           :  +0.50 (encouraged to try)")
print("  think + correct boxed         : +10.50 (maximum signal)")
print()
print("  Critical ordering enforced:")
print("    correct > tried_wrong > no_answer > no_thinking")
print("    Previously: doing_nothing > tried_wrong (was inverted)")

## Step 3: VRAM Safety Check Before BAPO Training

training is more VRAM-hungry than SFT because it holds **G=6 completions** in memory
simultaneously during the rollout phase. We check free VRAM before launching and
abort with an actionable error if headroom is insufficient.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# 1. Fetch the GPU Hardware Stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
free_memory = max_memory - start_gpu_memory

print(f"\n GPU: {gpu_stats.name}")
print(f"   Total VRAM:    {max_memory} GB")
print(f"   Reserved VRAM: {start_gpu_memory} GB")
print(f"   Free VRAM:     {free_memory:.2f} GB\n")

is_critical = free_memory < 4.0
if is_critical:
    print(" WARNING: Less than 4GB free VRAM. Training may OOM.")
    print("   Consider reducing MAX_SEQ_LEN or LoRA rank.")

# ==========================================
# 2. SEABORN VRAM HUD GRAPH
# ==========================================
sns.set_theme(style="dark", rc={
    "axes.facecolor": "#121212", 
    "figure.facecolor": "#121212",
    "text.color": "white"
})

# Create a DataFrame for Seaborn to digest
df = pd.DataFrame({
    'Component': ['VRAM'],
    'Used': [start_gpu_memory],
    'Total': [max_memory]
})

f, ax = plot.subplots(figsize=(8, 1.2))

# Plot the background bar (Total VRAM / Free Space) in dark gray
sns.barplot(x="Total", y="Component", data=df, color="#2a2a2a", ax=ax)

# Plot the foreground bar (Used VRAM) overriding the color based on danger levels
color_reserved = '#ff3333' if is_critical else '#00e5ff' # Red if critical, Cyan if safe
sns.barplot(x="Used", y="Component", data=df, color=color_reserved, ax=ax)

# Clean up the axes for the pure HUD look
ax.set(ylabel="", xlabel="")
ax.set_xticks([])
ax.set_yticks([])
sns.despine(left=True, bottom=True)

# Add dynamic text overlays right inside the bars
if start_gpu_memory > 2.0:
    plot.text(start_gpu_memory / 2, 0, f"USED: {start_gpu_memory:.1f} GB", 
             color='#121212', va='center', ha='center', fontweight='bold', fontsize=10)
    
if free_memory > 2.0:
    plot.text(start_gpu_memory + (free_memory / 2), 0, f"FREE: {free_memory:.1f} GB", 
             color='#888888', va='center', ha='center', fontweight='bold', fontsize=10)

# THE FIX: Removed letterspacing, manually spaced the title for the cinematic look
plot.title("S Y S T E M   M E M O R Y   S T A T U S", color='white', fontsize=11, pad=10, fontweight='bold')
plot.tight_layout()
plot.show()

## Step 4: Configure + Training The Custom BAPO Trainer

Here we are using our **compute loss function** and implementing the core BAPO logic in Hugging Face's `GRPOTrainer`and`GRPOConfig` to intercept the loss calculation. We inject the asymmetric clip-higher logic and the bounded sampling filter to brutally optimize our Kaggle compute window. Moreover, implementing a custom KL penalty function to make the overall pipeline computationally efficient

In [ ]:
import importlib.util
import os
if not hasattr(importlib.util, "_orig_find_spec_bapo"):
    importlib.util._orig_find_spec_bapo = importlib.util.find_spec
    
    def _patched_find_spec(name, package=None):
        if name == "wandb":
            return None
        return importlib.util._orig_find_spec_bapo(name, package)
    
    importlib.util.find_spec = _patched_find_spec

from dataclasses import dataclass, field
from typing import Any, Optional
from trl.trainer.base_config import _BaseConfig

@dataclass
class BAPOConfig(_BaseConfig):
    # Core Training & Generation
    learning_rate: float = field(default=2e-5)
    num_generations: int = field(default=6)
    max_completion_length: int = field(default=2048)
    temperature: float = field(default=0.9)
    top_p: float = field(default=0.95)
    generation_batch_size: Optional[int] = field(default=None)
    steps_per_generation: Optional[int] = field(default=None)
    
    # BAPO Specific Adaptive Bounds
    ratio_lower_start: float = field(default=0.6)
    ratio_lower_max: float = field(default=0.9)
    ratio_lower_step: float = field(default=0.05)
    ratio_upper_start: float = field(default=1.2)
    ratio_upper_max: float = field(default=3.0)
    ratio_upper_step: float = field(default=0.05)
    adv_ratio_target: float = field(default=0.5)
    bapo_eps: float = field(default=1e-8)
    clip_ratio_c: float = field(default=3.0)
    mask_truncated: bool = field(default=False)
    
    # Reward
    reward_weights: Optional[list[float]] = field(default=None)

    def __post_init__(self):
        super().__post_init__()
        num_processes = self.world_size
        
        if self.generation_batch_size is None and self.steps_per_generation is None:
            self.steps_per_generation = self.gradient_accumulation_steps
            self.generation_batch_size = self.per_device_train_batch_size * num_processes * self.steps_per_generation
        elif self.generation_batch_size is not None and self.steps_per_generation is None:
            self.steps_per_generation = self.generation_batch_size // (self.per_device_train_batch_size * num_processes)
        elif self.generation_batch_size is None and self.steps_per_generation is not None:
            self.generation_batch_size = self.per_device_train_batch_size * num_processes * self.steps_per_generation
            
        if self.num_generations < 2:
            raise ValueError("BAPO requires at least 2 generations per prompt to calculate advantages.")

import time
import torch
import torch.nn.functional as F
from collections import defaultdict
from transformers import PreTrainedModel, PreTrainedTokenizerBase, GenerationConfig
from trl.trainer.base_trainer import _BaseTrainer
from trl.trainer.utils import RepeatSampler, pad, selective_log_softmax
from accelerate.utils import gather
import sys
import gc

class BAPOTrainer(_BaseTrainer):
    def __init__(self, model, reward_funcs, args, train_dataset, processing_class,callbacks=None):
        self.reward_funcs = reward_funcs if isinstance(reward_funcs, list) else [reward_funcs]
        self.reward_weights = torch.ones(len(self.reward_funcs), dtype=torch.float32) if args.reward_weights is None else torch.tensor(args.reward_weights, dtype=torch.float32)
        
        self.max_completion_length = args.max_completion_length
        self.num_generations = args.num_generations
        self.temperature = args.temperature
        self.top_p = args.top_p
        
        # BAPO bounds
        self.ratio_lower_start = args.ratio_lower_start
        self.ratio_lower_max = args.ratio_lower_max
        self.ratio_lower_step = args.ratio_lower_step
        self.ratio_upper_start = args.ratio_upper_start
        self.ratio_upper_max = args.ratio_upper_max
        self.ratio_upper_step = args.ratio_upper_step
        self.adv_ratio_target = args.adv_ratio_target
        self.bapo_eps = args.bapo_eps
        self.clip_ratio_c = getattr(args, "clip_ratio_c", 3.0)
        self.mask_truncated = getattr(args, "mask_truncated", False)
        
        self._step = 0
        self._buffered_inputs = None
        self._metrics = {"train": defaultdict(list)}
        self._stats = {"last_logged_step": -1}
        self.shuffle_dataset = True

        self._nemotron_cache_cls = _find_nemotron_cache(model)

        super().__init__(
            model=model,
            args=args,
            data_collator=lambda x: x,
            train_dataset=train_dataset,
            processing_class=processing_class,
            callbacks=callbacks,
            compute_loss_func="disable", 
        )
        self.model_accepts_loss_kwargs = False
        
        self.generation_config = GenerationConfig(
            max_new_tokens=self.max_completion_length,
            do_sample=True,
            pad_token_id=self.processing_class.pad_token_id,
            eos_token_id=self.processing_class.eos_token_id,
            temperature=self.temperature,
            top_p=self.top_p,
            repetition_penalty=1.0,
            disable_compile=True
        )

    def get_train_dataloader(self):
        return self._get_dataloader(
            dataset=self.train_dataset,
            description="Training",
            batch_size=self._train_batch_size * self.args.steps_per_generation,
            sampler_fn=self._get_train_sampler,
            is_training=True,
        )

    def _get_train_sampler(self, dataset=None):
        return RepeatSampler(
            data_source=dataset or self.train_dataset,
            mini_repeat_count=self.num_generations,
            batch_size=self.args.generation_batch_size // self.num_generations,
            repeat_count=self.args.steps_per_generation,
            shuffle=self.shuffle_dataset,
            seed=self.args.seed,
        )

    def _generate_completions(self, prompts):
        device = self.accelerator.device
        prompt_ids = self.processing_class(text=prompts)["input_ids"]
        prompt_tensors = [torch.tensor(ids) for ids in prompt_ids]
        padded_ids = pad(prompt_tensors, padding_value=self.processing_class.pad_token_id, padding_side="left")
        attention_mask = pad([torch.ones_like(t) for t in prompt_tensors], padding_value=0, padding_side="left")
        
        input_ids = padded_ids.to(device)
        attention_mask = attention_mask.to(device)
        batch_size = input_ids.size(0)
        prompt_length = input_ids.size(1)
        
        self.model.eval()
        self.model.config.use_cache = True
        
        lm_head = getattr(self.model, "lm_head", None)
        if lm_head is None and hasattr(self.model, "base_model"):
            lm_head = getattr(self.model.base_model, "lm_head", None)
            
        # Retain _inner for extracting the config for the custom cache
        _inner = self.model
        for _attr in ("base_model", "model", "model"):
            _inner = getattr(_inner, _attr, _inner)
        
        cache = None
        if getattr(self, "_nemotron_cache_cls", None) is not None:
            try:
                import inspect
                sig = inspect.signature(self._nemotron_cache_cls.__init__)
                params = [p for p in sig.parameters.keys() if p != "self"]
                cache_kwargs = {}
                for p in params:
                    pl = p.lower()
                    if "config" in pl:
                        cache_kwargs[p] = _inner.config
                    elif "batch" in pl:
                        cache_kwargs[p] = batch_size
                    elif "seq" in pl or "len" in pl or "max" in pl:
                        cache_kwargs[p] = prompt_length + self.max_completion_length
                    elif "device" in pl:
                        cache_kwargs[p] = device
                    elif "dtype" in pl:
                        cache_kwargs[p] = _inner.dtype
                cache = self._nemotron_cache_cls(**cache_kwargs)
            except Exception as e:
                cache = None
        
        eos_id = self.processing_class.eos_token_id
        pad_id = self.processing_class.pad_token_id
        max_new = self.max_completion_length
        temp = self.temperature
        top_p = self.top_p
        
        generated_tokens = []
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        cur_ids = input_ids  
        cur_mask = attention_mask
        cache_position = torch.arange(prompt_length, dtype=torch.long, device=device)
        
        with torch.no_grad():
            for step in range(max_new):
                fwd_kwargs = {
                    "input_ids": cur_ids,
                    "attention_mask": cur_mask,
                    "cache_position": cache_position,
                    "use_cache": True,
                }
                if cache is not None:
                    fwd_kwargs["cache_params"] = cache
                
                # Use _inner to bypass accelerate's FP32 wrapper which crashes on custom cache objects
                outputs = _inner(**fwd_kwargs)
                
                # If _inner is the base transformer, it returns hidden states instead of logits
                if hasattr(outputs, "logits"):
                    logits = outputs.logits[:, -1, :].float()
                else:
                    hidden_states = outputs[0]
                    logits = lm_head(hidden_states)[:, -1, :].float()
                
                if hasattr(outputs, "cache_params") and outputs.cache_params is not None:
                    cache = outputs.cache_params
                
                if temp > 0:
                    logits = logits / temp
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
                    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_mask = cumulative_probs - torch.softmax(sorted_logits, dim=-1) >= top_p
                    sorted_logits[sorted_mask] = float("-inf")
                    logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)
                    probs = torch.softmax(logits, dim=-1)
                    next_token = torch.multinomial(probs, num_samples=1).squeeze(-1)
                else:
                    next_token = logits.argmax(dim=-1)
                
                next_token[finished] = pad_id
                finished = finished | (next_token == eos_id)
                generated_tokens.append(next_token)
                
                cur_ids = next_token.unsqueeze(1)
                cache_position = torch.tensor([prompt_length + step], dtype=torch.long, device=device)
                cur_mask = torch.cat([cur_mask, torch.ones(batch_size, 1, device=device, dtype=cur_mask.dtype)], dim=1)
                
                if finished.all():
                    break
        
        self.model.train()
        self.model.config.use_cache = False
        
        completion_ids = torch.stack(generated_tokens, dim=1)
        is_eos = completion_ids == eos_id
        eos_idx = torch.full((batch_size,), completion_ids.size(1), dtype=torch.long, device=device)
        eos_idx[is_eos.any(dim=1)] = is_eos.int().argmax(dim=1)[is_eos.any(dim=1)]
        sequence_indices = torch.arange(completion_ids.size(1), device=device).expand(batch_size, -1)
        completion_mask = (sequence_indices <= eos_idx.unsqueeze(1)).int()
        
        completion_ids_list = [c[m].tolist() for c, m in zip(completion_ids.cpu(), completion_mask.bool().cpu())]
        completions = self.processing_class.batch_decode(completion_ids_list, skip_special_tokens=True)
        
        del cache
        torch.cuda.empty_cache()
        
        return prompt_ids, completion_ids_list, completions

    def _calculate_rewards(self, prompts, completions, completion_ids_list, answers):
        device = self.accelerator.device
        rewards_per_func = torch.zeros(len(completions), len(self.reward_funcs), device=device)
        
        for i, reward_func in enumerate(self.reward_funcs):
            output = reward_func(prompts=prompts, completions=completions, completion_ids=completion_ids_list, answer=answers)
            output = [r if r is not None else torch.nan for r in output]
            rewards_per_func[:, i] = torch.tensor(output, dtype=torch.float32, device=device)

        # length_bonuses = torch.tensor(
        #     [min(len(comp) / 10000.0, 0.05) for comp in completions],
        #     dtype=torch.float32, device=device
        # )

        if self.accelerator.num_processes > 1:
            rewards_per_func = gather(rewards_per_func)
            # length_bonuses = gather(length_bonuses)

        rewards = (rewards_per_func * self.reward_weights.to(device).unsqueeze(0)).nansum(dim=1)
        # rewards = rewards + length_bonuses
        
        # Save for tree logging in compute_loss
        self._last_rewards_per_func = rewards_per_func.detach().clone()
            
        grouped_rewards = rewards.view(-1, self.num_generations)
        mean_per_group = grouped_rewards.mean(dim=1)
        std_per_group = grouped_rewards.std(dim=1)

        zero_var_mask = (std_per_group < 1e-6)
        std_per_group_safe = std_per_group.clone()
        std_per_group_safe[zero_var_mask] = 1.0  

        mean_expanded = mean_per_group.repeat_interleave(self.num_generations)
        std_expanded = std_per_group_safe.repeat_interleave(self.num_generations)
        zero_var_expanded = zero_var_mask.repeat_interleave(self.num_generations)

        advantages = (rewards - mean_expanded) / (std_expanded + self.bapo_eps)
        advantages[zero_var_expanded] = 0.0
        
        if self.accelerator.is_main_process and getattr(self.state, "global_step", 1) % 15 == 0:
            if getattr(self, "_last_reward_printed_step", -1) != getattr(self.state, "global_step", 1):
                self._last_reward_printed_step = getattr(self.state, "global_step", 1)
                
                print("\n" + "="*70)
                print(f"| [REWARD DEBUG] Step {getattr(self.state, 'global_step', 1)} |")
                print("="*70)
                # Print just the first generation group to avoid spamming the console
                for i in range(min(self.num_generations, len(rewards))):
                    print(f"[Group 1 | Sample {i+1}]")
                    func_names = [f.__name__ if hasattr(f, "__name__") else f"Reward_{j}" for j, f in enumerate(self.reward_funcs)]
                    func_scores = rewards_per_func[i].tolist()
                    weights = self.reward_weights.tolist()
                    
                    for name, score, weight in zip(func_names, func_scores, weights):
                        weighted = score * weight
                        print(f"  {name:25s} : {score:6.2f} x {weight:<4.1f} = {weighted:6.2f}")
                    
                    # print(f"  {'Length Bonus':25s} : {'-':>6} x {'-':<4} = {length_bonuses[i].item():6.2f}")
                    print(f"  >> Total Reward: {rewards[i].item():>8.4f}  |  Advantage: {advantages[i].item():>8.4f}")
                print("="*70 + "\n")
        
        process_slice = slice(
            self.accelerator.process_index * len(prompts), 
            (self.accelerator.process_index + 1) * len(prompts)
        )
        return advantages[process_slice]

    def _prepare_inputs(self, generation_batch):
        if self._step % self.args.steps_per_generation == 0 or self._buffered_inputs is None:
            prompts  = [item["prompt"]  for item in generation_batch]
            answers  = [item["answer"]  for item in generation_batch]  
            prompt_ids, completion_ids, completions = self._generate_completions(prompts)
            advantages = self._calculate_rewards(prompts, completions, completion_ids, answers)  
            
            device = self.accelerator.device
            
            # ── BUG 2 FIX: DATA INGESTION FILTER (Global Batch Level) ──────────────
            if len(advantages) >= self.num_generations:
                adv_grouped      = advantages.view(-1, self.num_generations)
                group_has_signal = adv_grouped.std(dim=1) > 1e-6
                if group_has_signal.any() and not group_has_signal.all():
                    signal_mask = group_has_signal.repeat_interleave(self.num_generations).float()
                    advantages  = advantages * signal_mask
            
            # ── PATH B FIX: GLOBAL BAPO DYNAMIC BOUNDS ─────────────────────────────
            #pos_adv = advantages.clamp(min=0).sum().item()
            # neg_adv = (-advantages.clamp(max=0)).sum().item()
            
            # clip_low = self.ratio_lower_start
            # clip_high = self.ratio_upper_start
            
            # iters = 0
            # MAX_ITERS = 20
            #     # Simulating the maximum clipping impact to find the perfect dynamic balance
            #     while iters < MAX_ITERS:
            #     current_rho = (pos_adv * clip_high) / (neg_adv * clip_low + self.bapo_eps)
            #         if current_rho >= self.adv_ratio_target:
            #             break
                    
            #         # If the scale is unbalanced, dynamically adjust the bounds
            #         if clip_low < self.ratio_lower_max:
            #             clip_low = min(clip_low + self.ratio_lower_step, self.ratio_lower_max)
            #         elif clip_high < self.ratio_upper_max:
            #             clip_high = min(clip_high + self.ratio_upper_step, self.ratio_upper_max)
            #         else:
            #             break
            #         iters += 1
                
            # # Convert calculated scalars into tensors so they can pass into the chunked batches safely
            # clip_low_tensor = torch.full((len(prompts),), clip_low, device=device)
            # Replace the entire bounds search block with fixed paper values
            clip_low  = self.ratio_lower_start   # 0.6
            clip_high = self.ratio_upper_start   # 1.2
            
            clip_low_tensor  = torch.full((len(prompts),), clip_low,  device=device)
            clip_high_tensor = torch.full((len(prompts),), clip_high, device=device)
            # ───────────────────────────────────────────────────────────────────────
            
            combined_ids = [torch.tensor(p + c, dtype=torch.long) for p, c in zip(prompt_ids, completion_ids)]
            input_ids_tensor = pad(combined_ids, padding_value=self.processing_class.pad_token_id, padding_side="right").to(device)
            attention_mask_tensor = (input_ids_tensor != self.processing_class.pad_token_id).long()
            
            seq_len = input_ids_tensor.shape[1]
            if seq_len % 8 != 0:
                pad_len = 8 - (seq_len % 8)
                input_ids_tensor = torch.nn.functional.pad(input_ids_tensor, (0, pad_len), value=self.processing_class.pad_token_id)
                attention_mask_tensor = torch.nn.functional.pad(attention_mask_tensor, (0, pad_len), value=0)
                seq_len = input_ids_tensor.shape[1]
            
            prompt_lengths = torch.tensor([len(p) for p in prompt_ids], device=device)
            completion_lengths = torch.tensor([len(c) for c in completion_ids], device=device)
            
            indices = torch.arange(seq_len, device=device).expand(len(combined_ids), -1)
            completion_mask = ((indices >= prompt_lengths.unsqueeze(1)) & 
                               (indices < (prompt_lengths + completion_lengths).unsqueeze(1))).long()
            
            if getattr(self.args, "mask_truncated", True):
                max_len = getattr(self, "max_completion_length", 2048)
                is_truncated = (completion_lengths >= max_len - 1)
                completion_mask[is_truncated] = 0

            # ── PATH B FIX: CHUNKED OLD_LOGPS (Mini-Batched for Speed) ─────────────────
            self.model.eval()
            _old_logps_list = []
            _olp_chunk = 3  # Mini-batch size: balances VRAM safety vs speed
            
            with torch.no_grad():
                for i in range(0, input_ids_tensor.size(0), _olp_chunk):
                    end_i = min(i + _olp_chunk, input_ids_tensor.size(0))
                    chunk_ids = input_ids_tensor[i:end_i]
                    chunk_mask = attention_mask_tensor[i:end_i]
                    
                    # use_cache=False is CRITICAL to prevent silent KV cache VRAM leaks
                    chunk_outputs = self.model(
                        input_ids=chunk_ids, attention_mask=chunk_mask, use_cache=False
                    )
                    chunk_logits = chunk_outputs.logits[:, :-1, :]
                    
                    chunk_logp = selective_log_softmax(
                        chunk_logits.float(), chunk_ids[:, 1:]
                    )
                    
                    _old_logps_list.append(chunk_logp)
                    del chunk_outputs, chunk_logits
                
                # Single cleanup after all chunks instead of per-sample
                torch.cuda.empty_cache()
                    
            _old_logps = torch.cat(_old_logps_list, dim=0)
            self.model.train()

            batch_dict = {
                "input_ids"           : input_ids_tensor,
                "attention_mask"      : attention_mask_tensor,
                "completion_mask"     : completion_mask,
                "advantages"          : advantages,
                "old_logps"           : _old_logps,        # The true anchor is back!
                "clip_low"            : clip_low_tensor,   
                "clip_high"           : clip_high_tensor,  
            }
            
            chunk_size = max(1, len(prompts) // self.args.steps_per_generation)
            self._buffered_inputs = []
            
            for i in range(0, len(prompts), chunk_size):
                end = min(i + chunk_size, len(prompts))
                self._buffered_inputs.append({k: v[i:end] for k, v in batch_dict.items()})
            
        return self._buffered_inputs[self._step % self.args.steps_per_generation]

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        input_ids       = inputs["input_ids"]
        attention_mask  = inputs["attention_mask"]
        completion_mask = inputs["completion_mask"][:, 1:]    
        advantages      = inputs["advantages"].unsqueeze(1)   

        # Retrieve the pre-calculated BAPO bounds for this specific chunk
        clip_low_bapo   = inputs["clip_low"].unsqueeze(1)
        clip_high_bapo  = inputs["clip_high"].unsqueeze(1)

        # ── 1. Forward Pass ────────────────────────────────────────────────────────
        logits    = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :]
        labels    = input_ids[:, 1:]
        new_logps = selective_log_softmax(logits.float(), labels)
        
        # Pulling the TRUE old_logps passed from the chunked generation above
        old_logps = inputs["old_logps"]

        # ── 2. The PPO Ratio ───────────────────────────────────────────────────────
        ratio = torch.exp(torch.clamp(new_logps - old_logps, -20.0, 20.0))

        # ── 3. Data Ingestion Filter ───────────────────────────────────────────────
        # Zero advantages for uninformative groups (all same reward) but preserve
        # completion_mask to keep the token-count denominator stable for all groups.
        G = self.num_generations
        B = advantages.shape[0]
        if B >= G and B % G == 0:
            adv_flat        = advantages.squeeze(1)               
            adv_grouped     = adv_flat.view(-1, G)                
            group_has_signal = adv_grouped.std(dim=1) > 1e-6      
            if group_has_signal.any() and not group_has_signal.all():
                signal_mask      = group_has_signal.repeat_interleave(G) 
                advantages       = advantages * signal_mask.float().unsqueeze(1)

        # ── 3.5 TRUE BAPO Dynamic Bounds (Calculated on actual ratios) ─────────────
        with torch.no_grad():
            clip_low = self.ratio_lower_start
            clip_high = self.ratio_upper_start
            iters = 0
            MAX_ITERS = 20
            
            mask_bool = completion_mask.bool()
            adv_expanded = advantages.expand_as(ratio)
            
            pos_mask = (adv_expanded > 0) & mask_bool
            neg_mask = (adv_expanded < 0) & mask_bool
            
            pos_A = adv_expanded[pos_mask]
            neg_A = adv_expanded[neg_mask]
            pos_r = ratio[pos_mask]
            neg_r = ratio[neg_mask]
            
            last_pos_obj = 0.0
            last_neg_obj = 0.0
            last_rho = 0.0
            
            while iters < MAX_ITERS:
                if len(pos_A) == 0 or len(neg_A) == 0:
                    break
                    
                pos_obj = torch.minimum(pos_r * pos_A, clip_high * pos_A).sum().item()
                neg_obj = torch.minimum(neg_r * neg_A, clip_low * neg_A).sum().item()
                
                last_pos_obj = pos_obj
                last_neg_obj = neg_obj
                
                # neg_obj is sum of negative numbers, so take absolute value
                current_rho = pos_obj / (abs(neg_obj) + self.bapo_eps)
                last_rho = current_rho
                
                if current_rho >= self.adv_ratio_target:
                    break
                    
                if clip_low < self.ratio_lower_max:
                    clip_low = min(clip_low + self.ratio_lower_step, self.ratio_lower_max)
                elif clip_high < self.ratio_upper_max:
                    clip_high = min(clip_high + self.ratio_upper_step, self.ratio_upper_max)
                else:
                    break
                iters += 1
                
        clip_low_bapo = torch.tensor(clip_low, device=ratio.device)
        clip_high_bapo = torch.tensor(clip_high, device=ratio.device)

        # ── 4. Applying the Dynamic BAPO Bounds ────────────────────────────────────
        clipped_ratio = torch.minimum(torch.maximum(ratio, clip_low_bapo), clip_high_bapo)

        # ── 5. Policy Optimization Loss ────────────────────────────────────────────
        pg_losses1      = -advantages * ratio           
        pg_losses2      = -advantages * clipped_ratio   
        clip_pg_losses1 = torch.maximum(pg_losses1, pg_losses2)  

        clip_ratio_c    = getattr(self, "clip_ratio_c", 3.0)
        pg_losses3      = -advantages * clip_ratio_c    
        clip_pg_losses2 = torch.minimum(pg_losses3, clip_pg_losses1)

        per_token_loss  = torch.where(
            advantages < 0, clip_pg_losses2, clip_pg_losses1
        ) * completion_mask

        # The DAPO Global Normalizer
        total_tokens = completion_mask.sum().clamp(min=1.0)
        policy_loss  = per_token_loss.sum() / total_tokens

        # ── 6. ZERO-VRAM KL PROXY (Anchored Weight-Space Regularization) ──────────
        # 6a. Lazy-load the snapshot on the first step
        # ── ZERO-VRAM KL PROXY (Fixed) ───────────────────────────────────────────────
        # Lazy snapshot on first call — captures initial SFT weights
        if not hasattr(self, "_ref_lora_weights"):
            self._ref_lora_weights = {}
            self._n_lora_params    = 0
            for name, param in model.named_parameters():
                if param.requires_grad and ('lora_A' in name or 'lora_B' in name):
                    self._ref_lora_weights[name] = param.detach().clone()
                    self._n_lora_params         += param.numel()
        
        # Raw squared drift — NO per-matrix normalisation (B matrices start at zero)
        # Raw L2 drift, no normalisation, beta tuned to raw lora_reg scale (~0.04)
        lora_reg = ratio.new_zeros(())
        for name, param in model.named_parameters():
            if param.requires_grad and ('lora_A' in name or 'lora_B' in name):
                ref      = self._ref_lora_weights[name]
                lora_reg = lora_reg + (param - ref).pow(2).sum()
        
        beta    = getattr(self.args, "beta", 5.0)   # 5.0 → kl_loss ≈ policy_loss at step 10
        kl_loss = beta * lora_reg
        loss    = policy_loss + kl_loss

        # ── 7. Mathematical Tracking & Logging ─────────────────────────────────────
        with torch.no_grad():
            kl_div_legacy = ((old_logps - new_logps) * completion_mask).sum() / total_tokens

        if self.accelerator.is_main_process and self._step > 0 and self._step % 5 == 0 and self._step != self._stats.get("last_logged_step", -1):
            self._stats["last_logged_step"] = self._step
            
            with torch.no_grad():
                valid_adv = advantages.expand_as(ratio)[completion_mask.bool()]
                valid_ratio = ratio[completion_mask.bool()]
                
                # --- Tree Log Computation ---
                rewards_cache = getattr(self, "_last_rewards_per_func", None)
                if rewards_cache is not None:
                    format_reward = rewards_cache[:, 0].mean().item()
                    correct_reward = rewards_cache[:, 2].mean().item()
                    hacker_reward = rewards_cache[:, 4].mean().item()
                    accuracy = (rewards_cache[:, 2] > 0).float().mean().item() * 100.0
                else:
                    format_reward = correct_reward = hacker_reward = accuracy = 0.0
                
                active_logps = new_logps[completion_mask.bool()]
                entropy = -active_logps.mean().item() if active_logps.numel() > 0 else 0.0
                avg_tokens = completion_mask.sum().item() / max(1, advantages.shape[0])
                adv_str = "[" + ",".join([f"{a:+.2f}" for a in advantages.squeeze(-1).tolist()[:4]]) + (",..." if len(advantages)>4 else "") + "]"
                
                print(f"\n[STEP {self._step}] Loss: {loss.item():.4f} | KL: {kl_loss.item():.4f}")
                print(f" ├─ Accuracy: {accuracy:.1f}% (Correct answers)")
                print(f" ├─ Rewards: Format={format_reward:.2f}, Correct={correct_reward:.2f}, Hacker={hacker_reward:.2f}")
                print(f" ├─ Stats: Tokens={avg_tokens:.0f}, Entropy={entropy:.2f}")
                print(f" ├─ Telemetry Sum: {completion_mask.sum().item()}, Bounds=[{clip_low_bapo.item():.2f},{clip_high_bapo.item():.2f}], KL Proxy={kl_loss.item():.6f}, Legacy KL={kl_div_legacy.item():.6f}")
                print(f" └─ Batch Adv: {adv_str}\n")
                
            if self._step % 15 == 0:
                with torch.no_grad():
                    adv_mean = valid_adv.mean().item() if len(valid_adv) > 0 else 0.0
                    adv_max = valid_adv.max().item() if len(valid_adv) > 0 else 0.0
                    adv_min = valid_adv.min().item() if len(valid_adv) > 0 else 0.0
                    
                    r_mean = valid_ratio.mean().item() if len(valid_ratio) > 0 else 0.0
                    r_max = valid_ratio.max().item() if len(valid_ratio) > 0 else 0.0
                    r_min = valid_ratio.min().item() if len(valid_ratio) > 0 else 0.0
                    
                    print("\n" + "="*70)
                print(f"| [BAPO MATH] Step {self._step} |")
                print("="*70)
                print(f"[1] ADVANTAGE SIGNAL (over valid tokens)")
                print(f"    Mean: {adv_mean:.4f}  |  Min: {adv_min:.4f}  |  Max: {adv_max:.4f}")
                print(f"\n[2] PPO RATIO (new_logp / old_logp)")
                print(f"    Mean: {r_mean:.4f}  |  Min: {r_min:.4f}  |  Max: {r_max:.4f}")
                print(f"\n[3] BAPO DYNAMIC BOUNDS SEARCH")
                print(f"    Iterations run  : {iters} / {MAX_ITERS}")
                print(f"    Final pos_obj   : {last_pos_obj:.4f} (Sum of positive updates)")
                print(f"    Final neg_obj   : {last_neg_obj:.4f} (Sum of negative updates)")
                print(f"    Current Rho     : {last_rho:.4f} (Target: >= {self.adv_ratio_target})")
                print(f"    Final clip_low  : {clip_low:.4f} (Started at {self.ratio_lower_start})")
                print(f"    Final clip_high : {clip_high:.4f} (Started at {self.ratio_upper_start})")
                print(f"\n[4] DAPO GLOBAL NORMALIZATION")
                print(f"    Total valid tokens in chunk: {total_tokens.item():.1f}")
                print(f"\n[5] FINAL LOSS COMPONENTS")
                print(f"    Policy Loss (Local speed limit) : {policy_loss.item():.4f}")
                print(f"    LoRA Drift Norm                 : {lora_reg.item():.4f}")
                print(f"    KL Penalty (Global anchor loss) : {kl_loss.item():.4f}")
                print(f"    Legacy KL (For comparison)      : {kl_div_legacy.item():.4f}")
                print(f"    Total Combined Loss             : {loss.item():.4f}")
                print("="*70 + "\n")

        self._step += 1
        return (loss, None) if return_outputs else loss


from transformers import TrainerCallback

# ── HARDWARE FLAGS ─────────────────────────────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()
print(f"[BAPO] VRAM before training: {torch.cuda.memory_reserved()/1024**3:.2f} GB reserved")


def _find_nemotron_cache(model_ref=None):
    """Multi-strategy discovery of the NemotronH cache class."""
    cls_name = "NemotronHHybridDynamicCache"

    def _patch_compat(cls):
        """Fix version mismatches between modeling code and cache class."""
        if cls is None or getattr(cls, "_bapo_compat_v3_applied", False):
            return cls
            
        # The model's modeling_nemotron_h.py calls update_conv_state(cache_init=True)
        # but older cache classes expect cache_position instead. Bridge the gap.
        orig_update = getattr(cls, "update_conv_state", None)
        if orig_update is not None:
            def _patched_update_conv_state(self, layer_idx, new_conv_state, cache_position=None, cache_init=False, **kwargs):
                if cache_init:
                    # Full initialization — write entire conv state directly
                    self.conv_states[layer_idx].copy_(new_conv_state)
                    return self.conv_states[layer_idx]
                return orig_update(self, layer_idx, new_conv_state, cache_position, **kwargs)
            cls.update_conv_state = _patched_update_conv_state
            print(f"[BAPO] Patched {cls.__name__}.update_conv_state for cache_init compat (v3)")
        
        # Ensure update_ssm_state exists (some older HF cache classes omit it entirely)
        orig_ssm = getattr(cls, "update_ssm_state", None)
        if orig_ssm is None:
            def _injected_update_ssm_state(self, layer_idx, new_ssm_state, cache_position=None, cache_init=False, **kwargs):
                # SSM state is sequence-agnostic (just recurrent state), so it's always a full copy
                self.ssm_states[layer_idx].copy_(new_ssm_state)
                return self.ssm_states[layer_idx]
            cls.update_ssm_state = _injected_update_ssm_state
            print(f"[BAPO] Injected missing {cls.__name__}.update_ssm_state (v3)")
        else:
            def _patched_update_ssm_state(self, layer_idx, new_ssm_state, cache_position=None, cache_init=False, **kwargs):
                if cache_init:
                    self.ssm_states[layer_idx].copy_(new_ssm_state)
                    return self.ssm_states[layer_idx]
                return orig_ssm(self, layer_idx, new_ssm_state, cache_position, **kwargs)
            cls.update_ssm_state = _patched_update_ssm_state
            print(f"[BAPO] Patched existing {cls.__name__}.update_ssm_state for cache_init compat (v3)")
            
        cls._bapo_compat_v3_applied = True
        return cls

    # PRIORITY 1: Extract from the exact module the model was loaded from
    if model_ref is not None:
        _inner = model_ref
        for attr in ("base_model", "model", "model"):
            _inner = getattr(_inner, attr, _inner)
        _model_module = type(_inner).__module__
        if _model_module:
            _mod = sys.modules.get(_model_module)
            if _mod is not None:
                _cc = getattr(_mod, cls_name, None)
                if _cc is not None:
                    print(f"[BAPO] {cls_name} found in EXACT model module ({_model_module})")
                    return _patch_compat(_cc)

    # PRIORITY 2: Direct Import Fallback
    try:
        from transformers.models.nemotron_h.modeling_nemotron_h import NemotronHHybridDynamicCache
        print(f"[BAPO] {cls_name} found via direct import.")
        return _patch_compat(NemotronHHybridDynamicCache)
    except ImportError:
        pass

    # PRIORITY 3: cache_utils
    try:
        import transformers.cache_utils as _cu
        _cc = getattr(_cu, cls_name, None)
        if _cc is not None:
            print(f"[BAPO] {cls_name} found in transformers.cache_utils.")
            return _patch_compat(_cc)
    except ImportError:
        pass

    # PRIORITY 4: brute-force sys.modules scan
    for _mod_name, _mod in list(sys.modules.items()):
        if _mod is None:
            continue
        _cc = getattr(_mod, cls_name, None)
        if _cc is not None:
            print(f"[BAPO] {cls_name} found via sys.modules brute-force in {_mod_name}.")
            return _patch_compat(_cc)

    print(f"[BAPO] WARNING: {cls_name} NOT found. Generation will be slow (O(L²)).")
    return None



# ─────────────────────────────────────────────────────────────────────────────
class KaggleTimeLimitCallback(TrainerCallback):
    """Measures from TRAINING STEP 1 — not notebook start."""
    def __init__(self, max_hours=10.0):
        self.max_seconds         = max_hours * 3600
        self.training_start_time = None

    def on_step_begin(self, args, state, control, **kwargs):
        if self.training_start_time is None:
            self.training_start_time = time.time()
            print(f"[TIMER] Training clock started. Budget: {self.max_seconds/3600:.1f}h")

    def on_step_end(self, args, state, control, **kwargs):
        if self.training_start_time is None:
            return
        elapsed = time.time() - self.training_start_time
        if elapsed >= self.max_seconds:
            print(f"\n[SAFETY] {elapsed/3600:.2f}h elapsed. Stopping and saving.")
            control.should_training_stop = True


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG FACTORY — per-run config with category-specific hyperparams
# ─────────────────────────────────────────────────────────────────────────────
REWARD_FUNCS = [
    reward_format,
    reward_thinking_quality,
    reward_correctness,
    reward_numerical,
    reward_no_hacking,
    reward_overlong_shaping,
]

REWARD_WEIGHTS = [
    2.0,  # reward_format (increased priority)
    1.0,  # reward_thinking_quality
    2.0,  # reward_correctness
    1.5,  # reward_numerical
    1.0,  # reward_no_hacking
    0.5,  # reward_overlong_shaping (decreased penalty)
]

def build_bapo_config(run_cfg: dict) -> BAPOConfig:
    """Build a BAPOConfig for one sequential BAPO run."""
    return BAPOConfig(
        output_dir                  = f"/kaggle/working/bapo_{run_cfg['name']}",
        max_completion_length       = 2048,
        num_generations             = 4,
        temperature                 = 0.75,
        top_p                       = 0.95,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        steps_per_generation        = 1,
        gradient_checkpointing      = True,
        optim                       = "paged_adamw_8bit",
        learning_rate               = run_cfg["lr"],
        max_steps                   = run_cfg["max_steps"],
        warmup_steps                = max(1, int(run_cfg["max_steps"] * 0.05)),
        lr_scheduler_type           = "cosine",
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        bf16                        = True,
        fp16                        = False,
        dataloader_pin_memory       = True,
        dataloader_num_workers      = 2,
        logging_steps               = 5,
        save_strategy               = "no",
        report_to                   = "none",
        seed                        = 3407,
        remove_unused_columns       = False,
        reward_weights              = REWARD_WEIGHTS,
    )


# ─────────────────────────────────────────────────────────────────────────────
# SEQUENTIAL BAPO TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
TOTAL_BUDGET_HOURS = 11.5
_global_start = time.time()

print(f"\n[BAPO] COMMENCING SEQUENTIAL TRAINING — {len(BAPO_RUNS)} runs")
print(f"[BAPO] Total budget: {TOTAL_BUDGET_HOURS}h\n")

for run_idx, run_cfg in enumerate(BAPO_RUNS):
    elapsed_h = (time.time() - _global_start) / 3600
    remaining_h = TOTAL_BUDGET_HOURS - elapsed_h
    if remaining_h < 0.5:
        print(f"\n[SAFETY] Only {remaining_h:.2f}h left. Skipping run '{run_cfg['name']}'.")
        break

    ds = bapo_run_datasets[run_cfg["name"]]
    print(f"\n{'='*60}")
    print(f"[BAPO RUN {run_idx+1}/{len(BAPO_RUNS)}] '{run_cfg['name']}'")
    print(f"  {run_cfg['description']}")
    print(f"  Samples: {len(ds)} | LR: {run_cfg['lr']:.0e} | Steps: {run_cfg['max_steps']}")
    print(f"  Time elapsed: {elapsed_h:.2f}h | Remaining: {remaining_h:.2f}h")
    print(f"{'='*60}")

    cfg = build_bapo_config(run_cfg)
    run_hours = min(remaining_h - 0.3, TOTAL_BUDGET_HOURS / len(BAPO_RUNS))

    trainer = BAPOTrainer(
        model            = model,
        args             = cfg,
        train_dataset    = ds,
        reward_funcs     = REWARD_FUNCS,
        processing_class = tokenizer,
        callbacks        = [KaggleTimeLimitCallback(max_hours=run_hours)],
    )

    trainer.train()

    # # Save after each run (disabled — final save captures everything)
    # save_path = f"/kaggle/working/bapo_{run_cfg['name']}_final"
    # model.save_pretrained(save_path)
    print(f"[BAPO] Run '{run_cfg['name']}' complete.")

    # Clean up trainer to free VRAM
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

# Save the final combined adapter
_final_path = "/kaggle/working/nemotron_bapo_adapter_final"
model.save_pretrained(_final_path, save_embedding_layers=False)
tokenizer.save_pretrained(_final_path)

total_elapsed = (time.time() - _global_start) / 3600
print(f"\n[BAPO] ALL RUNS COMPLETE in {total_elapsed:.2f}h")
print(f"[BAPO] Final adapter: {_final_path}")

# Section 3: Phase 3

In Phase 3, we execute the final submission script. This phase uses **Test-Time Compute** to break the 0.90 accuracy barrier within the 9-hour Kaggle limit.

### Key Architectures:
1. **vLLM PagedAttention:** Generates tokens up to 10x faster than standard PyTorch.
2. **Self-Consistency (Majority Voting):** Generates 16 parallel reasoning paths and takes the mode answer.
3. **LogProb Tie-Breaker:** Solves ties by analyzing the internal confidence (logprobs) of the `<think>` blocks.
4. **Forced Pondering:** Dynamically injects context rules into the prompt for hard subjects.

In [ ]:
# import os
# import gc
# import re
# import math
# import pandas as pd
# import numpy as np
# from collections import Counter, defaultdict
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import peft.import_utils
# import peft.tuners.lora.torchao
# peft.import_utils.is_torchao_available = lambda: False
# peft.tuners.lora.torchao.is_torchao_available = lambda: False
# from peft import PeftModel

# print("INITIATING PHASE 3: MULTI-PATH NATIVE INFERENCE ENGINE...")

# # ── Re-define _find_nemotron_cache for Cell 23 (self-contained inference) ─────
# # This is the same function from Cell 20; re-defined here so inference works
# # even if Cell 20 didn't run in this session (e.g. after kernel restart).
# if '_find_nemotron_cache' not in dir():
#     import sys as _sys

#     def _find_nemotron_cache(model_ref=None):
#         """Multi-strategy discovery of the NemotronH cache class."""
#         cls_name = "NemotronHHybridDynamicCache"

#         def _patch_compat(cls):
#             if cls is None or getattr(cls, "_bapo_compat_v3_applied", False):
#                 return cls
#             orig_update = getattr(cls, "update_conv_state", None)
#             if orig_update is not None:
#                 def _patched_update_conv_state(self, layer_idx, new_conv_state, cache_position=None, cache_init=False, **kwargs):
#                     if cache_init:
#                         self.conv_states[layer_idx].copy_(new_conv_state)
#                         return self.conv_states[layer_idx]
#                     return orig_update(self, layer_idx, new_conv_state, cache_position, **kwargs)
#                 cls.update_conv_state = _patched_update_conv_state
#                 print(f"[INFERENCE] Patched {cls.__name__}.update_conv_state for cache_init compat")
#             orig_ssm = getattr(cls, "update_ssm_state", None)
#             if orig_ssm is None:
#                 def _injected_update_ssm_state(self, layer_idx, new_ssm_state, cache_position=None, cache_init=False, **kwargs):
#                     self.ssm_states[layer_idx].copy_(new_ssm_state)
#                     return self.ssm_states[layer_idx]
#                 cls.update_ssm_state = _injected_update_ssm_state
#                 print(f"[INFERENCE] Injected missing {cls.__name__}.update_ssm_state")
#             else:
#                 def _patched_update_ssm_state(self, layer_idx, new_ssm_state, cache_position=None, cache_init=False, **kwargs):
#                     if cache_init:
#                         self.ssm_states[layer_idx].copy_(new_ssm_state)
#                         return self.ssm_states[layer_idx]
#                     return orig_ssm(self, layer_idx, new_ssm_state, cache_position, **kwargs)
#                 cls.update_ssm_state = _patched_update_ssm_state
#                 print(f"[INFERENCE] Patched {cls.__name__}.update_ssm_state for cache_init compat")
#             cls._bapo_compat_v3_applied = True
#             return cls

#         # PRIORITY 1: Extract from the exact module the model was loaded from
#         if model_ref is not None:
#             _inner = model_ref
#             for attr in ("base_model", "model", "model"):
#                 _inner = getattr(_inner, attr, _inner)
#             _model_module = type(_inner).__module__
#             if _model_module:
#                 _mod = _sys.modules.get(_model_module)
#                 if _mod is not None:
#                     _cc = getattr(_mod, cls_name, None)
#                     if _cc is not None:
#                         print(f"[INFERENCE] {cls_name} found in model module ({_model_module})")
#                         return _patch_compat(_cc)

#         # PRIORITY 2: Direct import
#         try:
#             from transformers.models.nemotron_h.modeling_nemotron_h import NemotronHHybridDynamicCache
#             print(f"[INFERENCE] {cls_name} found via direct import.")
#             return _patch_compat(NemotronHHybridDynamicCache)
#         except ImportError:
#             pass

#         # PRIORITY 3: cache_utils
#         try:
#             import transformers.cache_utils as _cu
#             _cc = getattr(_cu, cls_name, None)
#             if _cc is not None:
#                 print(f"[INFERENCE] {cls_name} found in transformers.cache_utils.")
#                 return _patch_compat(_cc)
#         except ImportError:
#             pass

#         # PRIORITY 4: brute-force sys.modules scan
#         for _mod_name, _mod in list(_sys.modules.items()):
#             if _mod is None:
#                 continue
#             _cc = getattr(_mod, cls_name, None)
#             if _cc is not None:
#                 print(f"[INFERENCE] {cls_name} found via sys.modules scan in {_mod_name}.")
#                 return _patch_compat(_cc)

#         print(f"[INFERENCE] WARNING: {cls_name} NOT found. Generation will be slow (O(L²)).")
#         return None

# # Purge Jupyter / IPython history to free up all strong references to old models
# try:
#     from IPython import get_ipython
#     ipy = get_ipython()
#     if ipy is not None:
#         print("[INFERENCE] Purging IPython output history to prevent VRAM leak...")
#         ipy.user_ns.pop('_', None)
#         ipy.user_ns.pop('__', None)
#         ipy.user_ns.pop('___', None)
#         if 'Out' in ipy.user_ns:
#             for k in list(ipy.user_ns['Out'].keys()):
#                 ipy.user_ns['Out'][k] = None
# except Exception as e:
#     pass

# for _obj_name in ("model", "trainer", "dapo_trainer", "bapo_trainer", "llm", "outputs"):
#     if _obj_name in globals():
#         del globals()[_obj_name]
# gc.collect()
# torch.cuda.empty_cache()
# print(f"[INFERENCE] VRAM freed: {torch.cuda.memory_reserved()/1024**3:.2f} GB reserved")

# BASE_MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
# ADAPTER_PATH    = "/kaggle/input/datasets/banwait13/lora-adapter/NEMOTRON/1"

# assert os.path.isdir(ADAPTER_PATH), f"Adapter not found at {ADAPTER_PATH}"

# class InferenceEngine:
#     def __init__(self, base_model_path, adapter_path, max_completion_length=1536):
#         self.max_completion_length = max_completion_length
#         print(f"[INFERENCE] Loading base model from {base_model_path}...")
#         self.tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
#         if self.tokenizer.unk_token is not None:
#             self.tokenizer.pad_token = self.tokenizer.unk_token
#         else:
#             self.tokenizer.pad_token = self.tokenizer.eos_token
#         self.tokenizer.padding_side = "left"
        
#         self.model = AutoModelForCausalLM.from_pretrained(
#             base_model_path,
#             device_map="auto",
#             torch_dtype=torch.bfloat16,
#             trust_remote_code=True,
#         )
        
#         print(f"[INFERENCE] Attaching adapter from {adapter_path}...")
#         self.model = PeftModel.from_pretrained(self.model, adapter_path, is_trainable=False)
#         self.model.eval()
        
#         # Discover Nemotron Cache class from the current model instance
#         # (same pattern as BAPOTrainer in Cell 20)
#         self._nemotron_cache_cls = _find_nemotron_cache(self.model)
        
#     def generate_single_path(self, prompt, temperature=0.6, top_p=0.95):
#         """Generate a single path using the custom cache-accelerated loop to guarantee stability and speed."""
#         device = next(self.model.parameters()).device
#         inputs = self.tokenizer(prompt, return_tensors="pt")
#         input_ids = inputs["input_ids"].to(device)
#         attention_mask = inputs["attention_mask"].to(device)
        
#         prompt_length = input_ids.size(1)
#         batch_size = 1
        
#         _inner = self.model
#         for _attr in ("base_model", "model", "model"):
#             _inner = getattr(_inner, _attr, _inner)
            
#         cache = None
#         if self._nemotron_cache_cls is not None:
#             try:
#                 import inspect
#                 sig = inspect.signature(self._nemotron_cache_cls.__init__)
#                 params = [p for p in sig.parameters.keys() if p != "self"]
#                 cache_kwargs = {}
#                 for p in params:
#                     pl = p.lower()
#                     if "config" in pl: cache_kwargs[p] = _inner.config
#                     elif "batch" in pl: cache_kwargs[p] = batch_size
#                     elif "seq" in pl or "len" in pl or "max" in pl: cache_kwargs[p] = prompt_length + self.max_completion_length
#                     elif "device" in pl: cache_kwargs[p] = device
#                     elif "dtype" in pl: cache_kwargs[p] = _inner.dtype
#                 cache = self._nemotron_cache_cls(**cache_kwargs)
#                 print(f"[INFERENCE] Cache initialized: {type(cache).__name__} args={list(cache_kwargs.keys())}")
#             except Exception as e:
#                 print(f"[INFERENCE] Cache init failed ({e}), proceeding without cache (slow).")
#                 cache = None
                
#         eos_id = self.tokenizer.eos_token_id
#         pad_id = self.tokenizer.pad_token_id
        
#         generated_tokens = []
#         logprobs = []
        
#         cur_ids = input_ids
#         cur_mask = attention_mask
#         cache_position = torch.arange(prompt_length, dtype=torch.long, device=device)
        
#         with torch.no_grad():
#             for step in range(self.max_completion_length):
#                 fwd_kwargs = {
#                     "input_ids": cur_ids,
#                     "attention_mask": cur_mask,
#                     "cache_position": cache_position,
#                     "use_cache": True,
#                 }
#                 if cache is not None:
#                     fwd_kwargs["cache_params"] = cache
                    
#                 outputs = _inner(**fwd_kwargs)
#                 logits = outputs.logits[:, -1, :].float()
                
#                 if hasattr(outputs, "cache_params") and outputs.cache_params is not None:
#                     cache = outputs.cache_params
                    
#                 if temperature > 0:
#                     scaled_logits = logits / temperature
#                     sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True, dim=-1)
#                     cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
#                     sorted_mask = cumulative_probs - torch.softmax(sorted_logits, dim=-1) >= top_p
#                     sorted_logits[sorted_mask] = float("-inf")
#                     scaled_logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)
#                     probs = torch.softmax(scaled_logits, dim=-1)
#                     next_token = torch.multinomial(probs, num_samples=1).squeeze(-1)
#                 else:
#                     next_token = logits.argmax(dim=-1)
                    
#                 lp = torch.log_softmax(logits, dim=-1)
#                 token_lp = lp[0, next_token[0]].item()
#                 logprobs.append(token_lp)
                
#                 generated_tokens.append(next_token)
#                 if next_token.item() == eos_id:
#                     break
                    
#                 cur_ids = next_token.unsqueeze(1)
#                 cache_position = torch.tensor([prompt_length + step], dtype=torch.long, device=device)
#                 cur_mask = torch.cat([cur_mask, torch.ones(1, 1, device=device, dtype=cur_mask.dtype)], dim=1)
                
#         completion_ids = torch.cat(generated_tokens) if generated_tokens else torch.tensor([], dtype=torch.long, device=device)
#         decoded = self.tokenizer.decode(completion_ids, skip_special_tokens=True) if len(completion_ids) > 0 else ""
        
#         del cache
#         torch.cuda.empty_cache()
        
#         mean_lp = float(np.mean(logprobs)) if logprobs else -999.0
#         return decoded, mean_lp

# engine = InferenceEngine(BASE_MODEL_PATH, ADAPTER_PATH, max_completion_length=1536)
# print("[INFERENCE] Native Nemotron inference engine online.")

In [ ]:
# _BOXED_RE = re.compile(r"\\boxed\{")

# def extract_boxed_inference(text: str) -> str:
#     """Extract the LAST \\boxed{...} with full nested-brace support."""
#     if not text:
#         return None
#     positions = [m.start() for m in _BOXED_RE.finditer(text)]
#     if not positions:
#         return None

#     start = positions[-1]
#     brace_start = start + len("\\boxed{")
#     depth = 1
#     i = brace_start
#     while i < len(text) and depth > 0:
#         if text[i] == "{":
#             depth += 1
#         elif text[i] == "}":
#             depth -= 1
#         i += 1

#     if depth != 0:
#         simple = re.search(r"\\boxed\{([^{}]*)\}", text)
#         return simple.group(1).strip() if simple else None

#     return text[brace_start:i-1].strip()

# def multi_fallback_extract(text: str) -> str:
#     """Multi-stage fallback extraction to ensure we never return an empty answer."""
#     if not text:
#         return None
#     ans = extract_boxed_inference(text)
#     if ans: return ans
    
#     match = re.search(r"Final Answer:\s*(.*?)(?:\n|$)", text, re.IGNORECASE)
#     if match: return match.group(1).strip()
    
#     nums = re.findall(r"=\s*(-?\d+(?:\.\d+)?)", text)
#     if nums: return nums[-1]
    
#     post_think = text.split("</think>")[-1] if "</think>" in text else text
#     nums = re.findall(r"-?\d+(?:\.\d+)?", post_think)
#     if nums: return nums[-1]
    
#     return None

# def _normalize_answer(s: str) -> str:
#     """Category-aware robust normalization."""
#     if s is None: return ""
#     s = s.strip()
#     s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
#     s = re.sub(r"\\[,;!]", "", s)
#     s = re.sub(r"\\frac\{([^}]+)\}\{([^}]+)\}", r"\1/\2", s)
#     s = re.sub(r"\s+", "", s).lower()
#     s = s.rstrip(".")
    
#     try:
#         if "/" in s:
#             num, den = s.split("/", 1)
#             f_val = float(num) / float(den)
#         else:
#             clean_s = s.replace("%", "").replace("$", "")
#             f_val = float(clean_s)
        
#         if f_val.is_integer():
#             return str(int(f_val))
#         else:
#             return f"{f_val:.4f}"
#     except Exception:
#         pass
        
#     return s

# def format_inference_prompt(question: str) -> str:
#     question_with_hint = (
#         question
#         + "\nThink step by step inside <think> tags, then give your final answer inside \\boxed{}."
#         + "\nFormat required: <think>your reasoning</think> \\boxed{your answer}"
#     )
#     return (
#         "<|im_start|>system\ndetailed thinking on<|im_end|>\n"
#         f"<|im_start|>user\n{question_with_hint}<|im_end|>\n"
#         "<|im_start|>assistant\n<think>\n"
#     )

In [ ]:
# gc.collect()
# torch.cuda.empty_cache()

# # 1. Fetch the GPU Hardware Stats
# gpu_stats = torch.cuda.get_device_properties(0)
# start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
# max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
# free_memory = max_memory - start_gpu_memory

# print(f"\n GPU: {gpu_stats.name}")
# print(f"   Total VRAM:    {max_memory} GB")
# print(f"   Reserved VRAM: {start_gpu_memory} GB")
# print(f"   Free VRAM:     {free_memory:.2f} GB\n")

# is_critical = free_memory < 4.0
# if is_critical:
#     print(" WARNING: Less than 4GB free VRAM. Training may OOM.")
#     print("   Consider reducing MAX_SEQ_LEN or LoRA rank.")

# # ==========================================
# # 2. SEABORN VRAM HUD GRAPH
# # ==========================================
# sns.set_theme(style="dark", rc={
#     "axes.facecolor": "#121212", 
#     "figure.facecolor": "#121212",
#     "text.color": "white"
# })

# # Create a DataFrame for Seaborn to digest
# df = pd.DataFrame({
#     'Component': ['VRAM'],
#     'Used': [start_gpu_memory],
#     'Total': [max_memory]
# })

# f, ax = plot.subplots(figsize=(8, 1.2))

# # Plot the background bar (Total VRAM / Free Space) in dark gray
# sns.barplot(x="Total", y="Component", data=df, color="#2a2a2a", ax=ax)

# # Plot the foreground bar (Used VRAM) overriding the color based on danger levels
# color_reserved = '#ff3333' if is_critical else '#00e5ff' # Red if critical, Cyan if safe
# sns.barplot(x="Used", y="Component", data=df, color=color_reserved, ax=ax)

# # Clean up the axes for the pure HUD look
# ax.set(ylabel="", xlabel="")
# ax.set_xticks([])
# ax.set_yticks([])
# sns.despine(left=True, bottom=True)

# # Add dynamic text overlays right inside the bars
# if start_gpu_memory > 2.0:
#     plot.text(start_gpu_memory / 2, 0, f"USED: {start_gpu_memory:.1f} GB", 
#              color='#121212', va='center', ha='center', fontweight='bold', fontsize=10)
    
# if free_memory > 2.0:
#     plot.text(start_gpu_memory + (free_memory / 2), 0, f"FREE: {free_memory:.1f} GB", 
#              color='#888888', va='center', ha='center', fontweight='bold', fontsize=10)

# # THE FIX: Removed letterspacing, manually spaced the title for the cinematic look
# plot.title("S Y S T E M   M E M O R Y   S T A T U S", color='white', fontsize=11, pad=10, fontweight='bold')
# plot.tight_layout()
# plot.show()

In [ ]:
# # Load Test Data
# test_df = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')

# question_col = next((col for col in ['problem', 'question', 'text', 'prompt', 'input'] if col in test_df.columns), None)
# if question_col is None:
#     non_id_cols = [c for c in test_df.columns if c.lower() != 'id']
#     question_col = non_id_cols[0] if non_id_cols else test_df.columns[0]

# id_col = next((col for col in test_df.columns if col.lower() == 'id'), 'id')
# print(f"[INFERENCE] Detected question column: '{question_col}', id column: '{id_col}'")
# questions_list = test_df[question_col].tolist()
# prompts = [format_inference_prompt(q) for q in questions_list]
# print(f"[INFERENCE] {len(prompts)} questions loaded.")

# def _build_reflexion_prompt(question: str, ans_a: str, ans_b: str) -> str:
#     return (
#         "<|im_start|>system\ndetailed thinking on<|im_end|>\n"
#         "<|im_start|>user\n"
#         f"You previously solved this problem and got two different answers: "
#         f"Answer A: \\boxed{{{ans_a}}} and Answer B: \\boxed{{{ans_b}}}.\n\n"
#         f"Original problem:\n{question}\n\n"
#         f"One of these answers contains a logical or computational error. "
#         f"Carefully re-derive the solution step by step. Identify which answer is wrong "
#         f"and why, then provide the correct final answer inside \\boxed{{}}.\n"
#         f"Format required: <think>your reasoning</think> \\boxed{{your answer}}"
#         "<|im_end|>\n"
#         "<|im_start|>assistant\n<think>\n"
#     )

# final_predictions = []
# extraction_stats  = {"extracted": 0, "fallback": 0, "greedy": 0, "reflexion": 0, "paths_generated": 0}

# for idx, (q, p) in enumerate(zip(questions_list, prompts)):
#     print(f"\n[INFERENCE] Q{idx+1}/{len(prompts)}: Generating adaptive paths...")
    
#     answers = []
#     norm_map = {}
#     confidences = defaultdict(list)
#     paths_generated = 0
    
#     def add_path(text, lp):
#         raw_ans = multi_fallback_extract(text)
#         if raw_ans is not None:
#             normed = _normalize_answer(raw_ans)
#             answers.append(normed)
#             confidences[normed].append(lp)
#             if normed not in norm_map:
#                 norm_map[normed] = raw_ans
#             return True
#         return False

#     # Stage 1: Easy Question Check (4 paths @ 0.6)
#     for _ in range(4):
#         text, lp = engine.generate_single_path(p, temperature=0.6, top_p=0.95)
#         add_path(text, lp)
#         paths_generated += 1
        
#     freq = Counter(answers)
#     if freq and freq.most_common(1)[0][1] == 4:
#         # 4/4 agreement! Easy question.
#         print("  -> Stage 1 consensus (4/4). Early exit.")
#     else:
#         # Stage 2: Medium Question Check (4 more paths @ 0.8)
#         print("  -> No Stage 1 consensus. Generating 4 more paths (T=0.8)...")
#         for _ in range(4):
#             text, lp = engine.generate_single_path(p, temperature=0.8, top_p=0.95)
#             add_path(text, lp)
#             paths_generated += 1
            
#         freq = Counter(answers)
#         if freq and freq.most_common(1)[0][1] >= 6:
#             # 6/8 agreement. Acceptable.
#             print(f"  -> Stage 2 consensus ({freq.most_common(1)[0][1]}/8). Early exit.")
#         else:
#             # Stage 3: Hard Question (24 more paths, mixed temp)
#             print("  -> Hard question detected. Generating 24 more paths (mixed temps)...")
#             mixed_temps = [0.4] * 6 + [0.6] * 6 + [0.8] * 6 + [1.0] * 6
#             for temp in mixed_temps:
#                 text, lp = engine.generate_single_path(p, temperature=temp, top_p=0.95)
#                 add_path(text, lp)
#                 paths_generated += 1
    
#     extraction_stats["paths_generated"] += paths_generated
    
#     if not answers:
#         print("  -> ALL paths failed extraction. Executing deterministic greedy fallback (T=0)...")
#         text, lp = engine.generate_single_path(p, temperature=0.0)
#         raw_ans = multi_fallback_extract(text)
#         if raw_ans is not None:
#             final_predictions.append(raw_ans)
#             extraction_stats["extracted"] += 1
#         else:
#             # Absolute worst-case scenario
#             final_predictions.append("0")
#             extraction_stats["greedy"] += 1
#         continue
        
#     extraction_stats["extracted"] += len(answers)
    
#     # Weighted Majority Voting
#     scored = {}
#     freq = Counter(answers)
#     for ans, count in freq.items():
#         # mean_logprob is negative, so exp(mean_logprob) is a probability [0, 1]
#         mean_prob = math.exp(float(np.mean(confidences[ans])))
#         scored[ans] = (count, mean_prob)
        
#     ranked = sorted(scored.items(), key=lambda x: (x[1][0], x[1][1]), reverse=True)
    
#     # Reflexion Tie-Breaker
#     if len(ranked) >= 2 and (ranked[0][1][0] - ranked[1][1][0]) <= 1:
#         print("  -> Tie-breaker triggered! Generating 3 reflexion paths (T=0.3)...")
#         ans_a_raw = norm_map[ranked[0][0]]
#         ans_b_raw = norm_map[ranked[1][0]]
#         ref_prompt = _build_reflexion_prompt(q, ans_a_raw, ans_b_raw)
        
#         ref_answers = []
#         for _ in range(3):
#             text, lp = engine.generate_single_path(ref_prompt, temperature=0.3)
#             r_ans = multi_fallback_extract(text)
#             if r_ans:
#                 n_ans = _normalize_answer(r_ans)
#                 ref_answers.append(n_ans)
#                 if n_ans not in norm_map:
#                     norm_map[n_ans] = r_ans
                    
#         if ref_answers:
#             ref_freq = Counter(ref_answers)
#             best_ref = ref_freq.most_common(1)[0][0]
#             final_predictions.append(norm_map[best_ref])
#             extraction_stats["reflexion"] += 1
#             print(f"  -> Reflexion selected: {norm_map[best_ref]}")
#             continue
            
#     best_normed = ranked[0][0]
#     final_predictions.append(norm_map[best_normed])
#     print(f"  -> Selected answer: {norm_map[best_normed]} (Votes: {ranked[0][1][0]})")

# print(f"\n[INFERENCE] Extraction stats:")
# print(f"  Successful Extractions: {extraction_stats['extracted']}")
# print(f"  Greedy Fallbacks:       {extraction_stats['greedy']}")
# print(f"  Reflexion Invocations:  {extraction_stats['reflexion']}")
# print(f"  Total Paths Generated:  {extraction_stats['paths_generated']}")

# # ── Create Submission ─────────────────────────────────────────────────────────
# submission = pd.DataFrame({
#     'id': test_df[id_col],
#     'answer': final_predictions
# })
# submission.to_csv('submission.csv', index=False)

# n_empty = sum(1 for a in final_predictions if a in ("0", "", None))
# print(f"\n[INFERENCE] submission.csv generated: {len(final_predictions)} predictions")
# print(f"  Empty/fallback answers: {n_empty}/{len(final_predictions)}")
# if n_empty > len(final_predictions) * 0.1:
#     print("  WARNING: >10% empty answers — model may need more training or longer max_tokens.")
# else:
#     print("  Quality check PASSED.")


# Section 3: Submission

The trained LoRA adapter is packaged into a `submission.zip` file. The competition evaluator loads the adapter on top of the base Nemotron model using vLLM and PEFT, so only the adapter files and tokenizer files are required. Required files that are missing will cause submission failure and are flagged explicitly.

In [ ]:
import zipfile
import shutil

# ── CHANGE 7: Submit the surgically fixed adapter, not the raw GRPO one ───
SUBMISSION_ZIP  = "/kaggle/working/submission.zip"
ADAPTER_SAVE_PATH ="/kaggle/working/nemotron_bapo_adapter_final"


REQUIRED_FILES = [
    "adapter_model.safetensors",
    "adapter_config.json",
]
TOKENIZER_FILES = [
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]

print("\n Creating submission.zip from surgically fixed adapter...")

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    packed = 0
    for fname in REQUIRED_FILES + TOKENIZER_FILES:
        fpath = os.path.join(ADAPTER_SAVE_PATH, fname)
        if os.path.exists(fpath):
            zf.write(fpath, arcname=fname)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"    Packed: {fname} ({size_mb:.1f} MB)")
            packed += 1
        else:
            if fname in REQUIRED_FILES:
                print(f"    MISSING REQUIRED: {fname} — submission will FAIL!")
            else:
                print(f"    Skipped (optional): {fname}")

zip_size = os.path.getsize(SUBMISSION_ZIP) / (1024 * 1024)
disk_free = shutil.disk_usage("/kaggle/working").free / (1024**3)

print(f"\n submission.zip ready: {zip_size:.1f} MB")
print(f"   Files packed  : {packed}")
print(f"   Disk remaining: {disk_free:.1f} GB")
print(f"   Location      : {SUBMISSION_ZIP}")
print(" ALL OPERATIONS COMPLETE. READY FOR SUBMISSION.")
